In [ ]:
# Setup: this notebook reads/writes relative to the repo's data/ folder.
# Run it from anywhere inside the repo; it switches to <repo>/data automatically.
import os
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data').is_dir() and (p / '2SLS').is_dir())
os.chdir(root / 'data')
print('Working directory:', Path.cwd())


# APPENDIX 1: DATA FETCH NOTEBOOK
### API FETCH: CONSUMPTION Denmark Full year

In [2]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.energidataservice.dk/dataset/ConsumptionConsumerCategoryHour"

out_dir = Path("DK_consumption_2025")
out_dir.mkdir(parents=True, exist_ok=True)

def fetch_consumption_month(year: int, month: int) -> Path:
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)

    params = {
        "start": start.strftime("%Y-%m-%dT%H:%M"),
        "end":   end.strftime("%Y-%m-%dT%H:%M"),
        "limit": 100000
    }

    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data.get("records", []))
    if df.empty:
        raise ValueError(f"No data returned for {year}-{month:02d}")

    df["TimeUTC"] = pd.to_datetime(df["TimeUTC"], utc=True)
    df = df.sort_values(["TimeUTC", "RegionName"]).reset_index(drop=True)

    path = out_dir / f"DKconsumption25_{month:02d}.parquet"
    df.to_parquet(path, index=False)
    return path

for month in range(1, 13):
    p = fetch_consumption_month(2025, month)
    print(f"Saved: {p}  |  Rows: {len(pd.read_parquet(p))}")

Saved: <project>/data/DK_consumption_2025/DKconsumption25_01.parquet  |  Rows: 11160
Saved: <project>/data/DK_consumption_2025/DKconsumption25_02.parquet  |  Rows: 10080
Saved: <project>/data/DK_consumption_2025/DKconsumption25_03.parquet  |  Rows: 11145
Saved: <project>/data/DK_consumption_2025/DKconsumption25_04.parquet  |  Rows: 10800
Saved: <project>/data/DK_consumption_2025/DKconsumption25_05.parquet  |  Rows: 11160
Saved: <project>/data/DK_consumption_2025/DKconsumption25_06.parquet  |  Rows: 10800
Saved: <project>/data/DK_consumption_2025/DKconsumption25_07.parquet  |  Rows: 11160
Saved: <project>/data/DK_consumption_2025/DKconsumption25_08.parquet  |  Rows: 11160
Saved: <project>/data/DK_consumption_2025/DKconsumption25_09.parquet  |  Rows: 10800
Saved: <project>/data/DK_consumption_2025/DKconsumption25_10.parquet  |  Rows: 11175
Saved: <project>/data/DK_consumption_2025/DKconsumption25_11.parquet  |  Rows: 10800
Saved: <project>/data/DK_consumption_2025/DKconsumption25_12.parq

### API FETCH: ELECTRICTY SPOT PRICES

In [11]:
import requests
import pandas as pd

URL = "https://api.energidataservice.dk/dataset/Elspotprices"

params = {
    "start": "2024-01-01T00:00",
    "end":   "2024-02-01T00:00",
    "filter": '{"PriceArea":["DK1","DK2"]}',
    "limit": 50000
}

r = requests.get(URL, params=params, timeout=60)
r.raise_for_status()

data = r.json()
df_month = pd.DataFrame(data["records"])

# Clean timestamps
df_month["HourUTC"] = pd.to_datetime(df_month["HourUTC"], utc=True)
df_month["HourDK"] = pd.to_datetime(df_month["HourDK"])

df_month = df_month.sort_values("HourUTC").reset_index(drop=True)

print("Rows:", len(df_month))
display(df_month.head())

Rows: 1488


,HourUTC,HourDK,PriceArea,SpotPriceDKK,SpotPriceEUR
0,2023-12-31 23:00:00+00:00,2024-01-01 00:00:00,DK2,217.160004,29.129999
1,2023-12-31 23:00:00+00:00,2024-01-01 00:00:00,DK1,126.660004,16.990000
2,2024-01-01 00:00:00+00:00,2024-01-01 01:00:00,DK1,209.779999,28.139999
3,2024-01-01 00:00:00+00:00,2024-01-01 01:00:00,DK2,212.160004,28.459999
4,2024-01-01 01:00:00+00:00,2024-01-01 02:00:00,DK1,198.740005,26.660000


In [ ]:
import requests
import pandas as pd
from pathlib import Path
import json

BASE_URL = "https://api.energidataservice.dk/dataset/Elspotprices"

out_dir = Path("raw/elspotprices_monthly")
out_dir.mkdir(parents=True, exist_ok=True)

def fetch_elspot_month(year: int, month: int, areas=("DK1", "DK2")) -> Path:
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)

    filter_obj = {"PriceArea": list(areas)}  

    params = {
        "start": start.strftime("%Y-%m-%dT%H:%M"),
        "end":   end.strftime("%Y-%m-%dT%H:%M"),
        "filter": json.dumps(filter_obj),     
        "limit": 100000
    }

    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data.get("records", []))
    if df.empty:
        raise ValueError(f"No data returned for {year}-{month:02d}")

    df["HourUTC"] = pd.to_datetime(df["HourUTC"], utc=True)
    df["HourDK"] = pd.to_datetime(df["HourDK"])
    df = df.sort_values(["HourUTC", "PriceArea"]).reset_index(drop=True)

    path = out_dir / f"elspot_{year}_{month:02d}.parquet"
    df.to_parquet(path, index=False)
    return path

# test: February 2024
p = fetch_elspot_month(2024, 2)
print("Saved:", p)
print("Rows:", len(pd.read_parquet(p)))

Saved: data/elspotprices_monthly/elspot_2024_02.parquet
Rows: 1392


In [13]:
from tqdm import tqdm

def fetch_elspot_year(year: int, areas=("DK1","DK2")):
    saved = []
    for m in tqdm(range(1, 13), desc=f"Elspot {year}"):
        path = out_dir / f"elspot_{year}_{m:02d}.parquet"
        if path.exists():
            continue
        saved.append(fetch_elspot_month(year, m, areas=areas))
    return saved

# test: download all of 2024
saved_2024 = fetch_elspot_year(2024)
print("New files saved:", len(saved_2024))

Elspot 2024: 100%|██████████| 12/12 [00:00<00:00, 9765.55it/s]

New files saved: 0


In [14]:
for y in range(2021, 2026):
    fetch_elspot_year(y)

print("Done.")

Elspot 2025:  75%|███████▌  | 9/12 [00:00<00:00, 64.25it/s]


ValueError: No data returned for 2025-10

In [ ]:
import pandas as pd
from pathlib import Path

total = 0

for y in range(2021, 2026):
    files = sorted(Path("raw/elspotprices_monthly").glob(f"elspot_{y}_*.parquet"))
    for f in files:
        total += len(pd.read_parquet(f))

print("TOTAL 2021–2025:", total)

##### Diagnostics

In [20]:
# DATA MANUAL — Elspotprices (Hourly spot prices DK1/DK2)
# Prints: file coverage, columns, dtypes, missingness, uniqueness, and basic sanity checks.

import pandas as pd
from pathlib import Path

DATASET_NAME = "Elspotprices (Hourly spot prices, DK1/DK2)"
DATA_DIR = Path("raw/elspotprices_monthly")
PATTERN = "elspot_*.parquet"

files = sorted(DATA_DIR.glob(PATTERN))
assert files, f"No files found in {DATA_DIR} matching {PATTERN}"

# --- Load one sample file for schema checks ---
sample_file = files[0]
df_sample = pd.read_parquet(sample_file)

print("="*80)
print("DATASET:", DATASET_NAME)
print("DIRECTORY:", DATA_DIR)
print("FILES:", len(files))
print("SAMPLE FILE:", sample_file.name)
print("="*80)

# --- Schema ---
print("\nCOLUMNS:")
for c in df_sample.columns:
    print(" -", c)

print("\nDTYPES (sample file):")
print(df_sample.dtypes)

# --- Coverage across all files (min/max time, total rows) ---
total_rows = 0
tmin, tmax = None, None

for f in files:
    df_t = pd.read_parquet(f, columns=["HourUTC"])
    total_rows += len(df_t)
    mn = df_t["HourUTC"].min()
    mx = df_t["HourUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("\nCOVERAGE:")
print(" - Total rows (all files):", total_rows)
print(" - Start HourUTC:", tmin)
print(" - End HourUTC:", tmax)

# --- Key integrity checks on sample file ---
print("\nINTEGRITY CHECKS (sample file):")

key_cols = ["HourUTC", "PriceArea"]
dup_keys = df_sample.duplicated(key_cols).sum()
print(" - Duplicate (HourUTC, PriceArea) rows:", int(dup_keys))

# PriceArea values
if "PriceArea" in df_sample.columns:
    print(" - PriceArea unique values:", sorted(df_sample["PriceArea"].dropna().unique()))

# Missingness
print("\nMISSINGNESS (sample file):")
na_counts = df_sample.isna().sum()
for c in df_sample.columns:
    print(f" - {c}: {int(na_counts[c])} missing")

# Price sanity checks
for col in ["SpotPriceDKK", "SpotPriceEUR"]:
    if col in df_sample.columns:
        s = df_sample[col]
        print(f"\nBASIC STATS ({col}) (sample file):")
        print(" - min:", float(s.min()))
        print(" - max:", float(s.max()))
        print(" - mean:", float(s.mean()))
        print(" - zeros:", int((s == 0).sum()))
        print(" - negatives:", int((s < 0).sum()))
        print(" - missing:", int(s.isna().sum()))

# Rows per hour sanity check (sample file) -> should be ~2 (DK1, DK2)
rows_per_hour = df_sample.groupby("HourUTC").size()
print("\nROWS PER HOUR (sample file):")
print(" - min:", int(rows_per_hour.min()))
print(" - max:", int(rows_per_hour.max()))
print(" - value counts:")
display(rows_per_hour.value_counts().head(10))

# --- Show a snippet ---
print("\nSNIPPET (first 10 rows of sample file):")
display(df_sample.head(10))

DATASET: Elspotprices (Hourly spot prices, DK1/DK2)
DIRECTORY: data/elspotprices_monthly
FILES: 57
SAMPLE FILE: elspot_2021_01.parquet

COLUMNS:
 - HourUTC
 - HourDK
 - PriceArea
 - SpotPriceDKK
 - SpotPriceEUR

DTYPES (sample file):
HourUTC         datetime64[ns, UTC]
HourDK               datetime64[ns]
PriceArea                    object
SpotPriceDKK                float64
SpotPriceEUR                float64
dtype: object

COVERAGE:
 - Total rows (all files): 83230
 - Start HourUTC: 2020-12-31 23:00:00+00:00
 - End HourUTC: 2025-09-30 21:00:00+00:00

INTEGRITY CHECKS (sample file):
 - Duplicate (HourUTC, PriceArea) rows: 0
 - PriceArea unique values: ['DK1', 'DK2']

MISSINGNESS (sample file):
 - HourUTC: 0 missing
 - HourDK: 0 missing
 - PriceArea: 0 missing
 - SpotPriceDKK: 0 missing
 - SpotPriceEUR: 0 missing

BASIC STATS (SpotPriceDKK) (sample file):
 - min: 0.15
 - max: 818.349976
 - mean: 375.9053559670699
 - zeros: 0
 - negatives: 0
 - missing: 0

BASIC STATS (SpotPriceEUR) (sa

2    744
Name: count, dtype: int64


SNIPPET (first 10 rows of sample file):


,HourUTC,HourDK,PriceArea,SpotPriceDKK,SpotPriceEUR
0,2020-12-31 23:00:00+00:00,2021-01-01 00:00:00,DK1,378.519989,50.869999
1,2020-12-31 23:00:00+00:00,2021-01-01 00:00:00,DK2,378.519989,50.869999
2,2021-01-01 00:00:00+00:00,2021-01-01 01:00:00,DK1,358.579987,48.189999
3,2021-01-01 00:00:00+00:00,2021-01-01 01:00:00,DK2,358.579987,48.189999
4,2021-01-01 01:00:00+00:00,2021-01-01 02:00:00,DK1,332.459991,44.680000
5,2021-01-01 01:00:00+00:00,2021-01-01 02:00:00,DK2,332.459991,44.680000
6,2021-01-01 02:00:00+00:00,2021-01-01 03:00:00,DK1,319.369995,42.919998
7,2021-01-01 02:00:00+00:00,2021-01-01 03:00:00,DK2,319.369995,42.919998
8,2021-01-01 03:00:00+00:00,2021-01-01 04:00:00,DK1,300.540009,40.389999
9,2021-01-01 03:00:00+00:00,2021-01-01 04:00:00,DK2,300.540009,40.389999


### API FETCH: Industry Consumption


In [21]:
import requests
import pandas as pd
import json

URL = "https://api.energidataservice.dk/dataset/ConsumptionConsumerCategoryHour"

start = "2024-01-01T00:00"
end   = "2024-02-01T00:00"

params = {
    "start": start,
    "end": end,
    "limit": 200000
}

r = requests.get(URL, params=params, timeout=60)
r.raise_for_status()

data = r.json()
df_month = pd.DataFrame(data["records"])

print("Rows:", len(df_month))
print("Columns:", list(df_month.columns))
display(df_month.head())

Rows: 11160
Columns: ['TimeUTC', 'TimeDK', 'RegionName', 'ConsumerCategory3', 'ConsumerCategory2', 'ConsumptionkWh']


,TimeUTC,TimeDK,RegionName,ConsumerCategory3,ConsumerCategory2,ConsumptionkWh
0,2024-01-31T22:00:00,2024-01-31T23:00:00,Region Hovedstaden,Erhverv,Erhverv,557854.984
1,2024-01-31T22:00:00,2024-01-31T23:00:00,Region Hovedstaden,Offentlige foretagender,Erhverv,86009.656
2,2024-01-31T22:00:00,2024-01-31T23:00:00,Region Hovedstaden,Privat,Privat,309868.309
3,2024-01-31T22:00:00,2024-01-31T23:00:00,Region Midtjylland,Erhverv,Erhverv,815907.522
4,2024-01-31T22:00:00,2024-01-31T23:00:00,Region Midtjylland,Offentlige foretagender,Erhverv,50367.948


In [22]:
from pathlib import Path
import pandas as pd

# parse timestamps
df_month["TimeUTC"] = pd.to_datetime(df_month["TimeUTC"], utc=True)
df_month["TimeDK"]  = pd.to_datetime(df_month["TimeDK"])

# sort (stable for later merges)
df_month = df_month.sort_values(["TimeUTC", "RegionName", "ConsumerCategory3", "ConsumerCategory2"]).reset_index(drop=True)

# save
out_dir = Path("raw/consumption_category_hourly_monthly")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "consumption_cat_hour_2024_01.parquet"
df_month.to_parquet(out_path, index=False)

print("Saved:", out_path)
print("Rows:", len(df_month))
display(df_month.head())

Saved: data/consumption_category_hourly_monthly/consumption_cat_hour_2024_01.parquet
Rows: 11160


,TimeUTC,TimeDK,RegionName,ConsumerCategory3,ConsumerCategory2,ConsumptionkWh
0,2023-12-31 23:00:00+00:00,2024-01-01,Region Hovedstaden,Erhverv,Erhverv,473621.920
1,2023-12-31 23:00:00+00:00,2024-01-01,Region Hovedstaden,Offentlige foretagender,Erhverv,79171.143
2,2023-12-31 23:00:00+00:00,2024-01-01,Region Hovedstaden,Privat,Privat,305809.198
3,2023-12-31 23:00:00+00:00,2024-01-01,Region Midtjylland,Erhverv,Erhverv,817448.194
4,2023-12-31 23:00:00+00:00,2024-01-01,Region Midtjylland,Offentlige foretagender,Erhverv,44773.775


In [23]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.energidataservice.dk/dataset/ConsumptionConsumerCategoryHour"

cons_dir = Path("raw/consumption_category_hourly_monthly")
cons_dir.mkdir(parents=True, exist_ok=True)

def fetch_consumption_month(year: int, month: int):
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)

    params = {
        "start": start.strftime("%Y-%m-%dT%H:%M"),
        "end":   end.strftime("%Y-%m-%dT%H:%M"),
        "limit": 500000
    }

    r = requests.get(BASE_URL, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data.get("records", []))
    if df.empty:
        return None  # ✅ skip silently (we log in the year loop)

    df["TimeUTC"] = pd.to_datetime(df["TimeUTC"], utc=True)
    df["TimeDK"]  = pd.to_datetime(df["TimeDK"])
    df = df.sort_values(["TimeUTC", "RegionName", "ConsumerCategory3", "ConsumerCategory2"]).reset_index(drop=True)

    path = cons_dir / f"consumption_cat_hour_{year}_{month:02d}.parquet"
    df.to_parquet(path, index=False)
    return path

def fetch_consumption_year(year: int):
    saved, skipped = 0, 0
    for m in range(1, 13):
        path = cons_dir / f"consumption_cat_hour_{year}_{m:02d}.parquet"
        if path.exists():
            continue

        out = fetch_consumption_month(year, m)
        if out is None:
            skipped += 1
            print(f"Skipped (no data): {year}-{m:02d}")
        else:
            saved += 1
    return saved, skipped

# download 2021–2025 (skips empty months)
saved_total, skipped_total = 0, 0
for y in range(2021, 2026):
    s, k = fetch_consumption_year(y)
    saved_total += s
    skipped_total += k

print("\nNew monthly files saved:", saved_total)
print("Months skipped (no data):", skipped_total)

Skipped (no data): 2021-01
Skipped (no data): 2021-02
Skipped (no data): 2021-03
Skipped (no data): 2021-04
Skipped (no data): 2021-05
Skipped (no data): 2021-06
Skipped (no data): 2021-07

New monthly files saved: 0
Months skipped (no data): 7


In [24]:
import pandas as pd
from pathlib import Path

files = sorted(Path("raw/consumption_category_hourly_monthly").glob("consumption_cat_hour_*.parquet"))

# concatenate just for min/max timestamp + total rows (not too big here)
total_rows = 0
tmin, tmax = None, None

for f in files:
    df = pd.read_parquet(f, columns=["TimeUTC"])
    total_rows += len(df)

    mn = df["TimeUTC"].min()
    mx = df["TimeUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("Files:", len(files))
print("Total rows:", total_rows)
print("Start TimeUTC:", tmin)
print("End TimeUTC:", tmax)

Files: 53
Total rows: 569940
Start TimeUTC: 2021-07-31 22:00:00+00:00
End TimeUTC: 2025-12-31 22:00:00+00:00


##### Diagnostics 

In [25]:
# DATA MANUAL — ConsumptionConsumerCategoryHour
# Prints: file coverage, columns, dtypes, missingness, uniqueness, and basic sanity checks.

import pandas as pd
from pathlib import Path

DATASET_NAME = "ConsumptionConsumerCategoryHour (Hourly consumption by region + consumer category)"
DATA_DIR = Path("raw/consumption_category_hourly_monthly")
PATTERN = "consumption_cat_hour_*.parquet"

files = sorted(DATA_DIR.glob(PATTERN))
assert files, f"No files found in {DATA_DIR} matching {PATTERN}"

# --- Load one sample file for schema checks ---
sample_file = files[0]
df_sample = pd.read_parquet(sample_file)

print("="*80)
print("DATASET:", DATASET_NAME)
print("DIRECTORY:", DATA_DIR)
print("FILES:", len(files))
print("SAMPLE FILE:", sample_file.name)
print("="*80)

# --- Schema ---
print("\nCOLUMNS:")
for c in df_sample.columns:
    print(" -", c)

print("\nDTYPES (sample file):")
print(df_sample.dtypes)

# --- Coverage across all files (min/max time, total rows) ---
total_rows = 0
tmin, tmax = None, None

for f in files:
    df_t = pd.read_parquet(f, columns=["TimeUTC"])
    total_rows += len(df_t)
    mn = df_t["TimeUTC"].min()
    mx = df_t["TimeUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("\nCOVERAGE:")
print(" - Total rows (all files):", total_rows)
print(" - Start TimeUTC:", tmin)
print(" - End TimeUTC:", tmax)

# --- Key integrity checks on sample file ---
print("\nINTEGRITY CHECKS (sample file):")

key_cols = ["TimeUTC", "RegionName", "ConsumerCategory3", "ConsumerCategory2"]
dup_keys = df_sample.duplicated(key_cols).sum()
print(" - Duplicate key rows:", int(dup_keys))

# Unique values (sample)
for col in ["RegionName", "ConsumerCategory3", "ConsumerCategory2"]:
    if col in df_sample.columns:
        vals = df_sample[col].dropna().unique()
        print(f" - {col} unique (sample): {len(vals)}")
        print("   example:", list(vals[:15]))

# Missingness
print("\nMISSINGNESS (sample file):")
na_counts = df_sample.isna().sum()
for c in df_sample.columns:
    print(f" - {c}: {int(na_counts[c])} missing")

# Consumption sanity checks
cons_col = "ConsumptionkWh"
if cons_col in df_sample.columns:
    s = df_sample[cons_col]
    print(f"\nBASIC STATS ({cons_col}) (sample file):")
    print(" - min:", float(s.min()))
    print(" - max:", float(s.max()))
    print(" - mean:", float(s.mean()))
    print(" - zeros:", int((s == 0).sum()))
    print(" - negatives:", int((s < 0).sum()))
    print(" - missing:", int(s.isna().sum()))

# Rows per hour sanity check (sample file)
rows_per_hour = df_sample.groupby("TimeUTC").size()
print("\nROWS PER HOUR (sample file):")
print(" - min:", int(rows_per_hour.min()))
print(" - max:", int(rows_per_hour.max()))
print(" - mode (most common):")
display(rows_per_hour.value_counts().head(10))

# --- Show a snippet ---
print("\nSNIPPET (first 10 rows of sample file):")
display(df_sample.head(10))

DATASET: ConsumptionConsumerCategoryHour (Hourly consumption by region + consumer category)
DIRECTORY: data/consumption_category_hourly_monthly
FILES: 53
SAMPLE FILE: consumption_cat_hour_2021_08.parquet

COLUMNS:
 - TimeUTC
 - TimeDK
 - RegionName
 - ConsumerCategory3
 - ConsumerCategory2
 - ConsumptionkWh

DTYPES (sample file):
TimeUTC              datetime64[ns, UTC]
TimeDK                    datetime64[ns]
RegionName                        object
ConsumerCategory3                 object
ConsumerCategory2                 object
ConsumptionkWh                   float64
dtype: object

COVERAGE:
 - Total rows (all files): 569940
 - Start TimeUTC: 2021-07-31 22:00:00+00:00
 - End TimeUTC: 2025-12-31 22:00:00+00:00

INTEGRITY CHECKS (sample file):
 - Duplicate key rows: 0
 - RegionName unique (sample): 5
   example: ['Region Hovedstaden', 'Region Midtjylland', 'Region Nordjylland', 'Region Sjælland', 'Region Syddanmark']
 - ConsumerCategory3 unique (sample): 3
   example: ['Erhverv', 'Of

15    744
Name: count, dtype: int64


SNIPPET (first 10 rows of sample file):


,TimeUTC,TimeDK,RegionName,ConsumerCategory3,ConsumerCategory2,ConsumptionkWh
0,2021-07-31 22:00:00+00:00,2021-08-01,Region Hovedstaden,Erhverv,Erhverv,389982.647
1,2021-07-31 22:00:00+00:00,2021-08-01,Region Hovedstaden,Offentlige foretagender,Erhverv,75472.830
2,2021-07-31 22:00:00+00:00,2021-08-01,Region Hovedstaden,Privat,Privat,173035.742
3,2021-07-31 22:00:00+00:00,2021-08-01,Region Midtjylland,Erhverv,Erhverv,362670.072
4,2021-07-31 22:00:00+00:00,2021-08-01,Region Midtjylland,Offentlige foretagender,Erhverv,44573.885
5,2021-07-31 22:00:00+00:00,2021-08-01,Region Midtjylland,Privat,Privat,153362.828
6,2021-07-31 22:00:00+00:00,2021-08-01,Region Nordjylland,Erhverv,Erhverv,217887.201
7,2021-07-31 22:00:00+00:00,2021-08-01,Region Nordjylland,Offentlige foretagender,Erhverv,21600.788
8,2021-07-31 22:00:00+00:00,2021-08-01,Region Nordjylland,Privat,Privat,78801.578
9,2021-07-31 22:00:00+00:00,2021-08-01,Region Sjælland,Erhverv,Erhverv,217637.322


### API FECTH: Consumption across industrial codes

In [35]:
import requests
import pandas as pd

URL = "https://api.energidataservice.dk/dataset/ConsumptionDK3619IndustryHour"

params = {
    "start": "2024-01-01T00:00",
    "end":   "2024-02-01T00:00",
    "limit": 500000
}

r = requests.get(URL, params=params, timeout=120)
r.raise_for_status()

data = r.json()
df_month = pd.DataFrame(data.get("records", []))

print("Rows:", len(df_month))
print("Columns:", list(df_month.columns))
display(df_month.head())

Rows: 24552
Columns: ['TimeUTC', 'TimeDK', 'DK36Code', 'DK36Title', 'DK19Code', 'DK19Title', 'Consumption_MWh']


,TimeUTC,TimeDK,DK36Code,DK36Title,DK19Code,DK19Title,Consumption_MWh
0,2024-01-31T22:00:00,2024-01-31T23:00:00,A,"Landbrug, skovbrug og fiskeri",A,"Landbrug, skovbrug og fiskeri",182.588459
1,2024-01-31T22:00:00,2024-01-31T23:00:00,B_E,Råstofindvinding & Vandforsyning og renovation,B_E,Råstofindvinding & Vandforsyning og renovation,172.636901
2,2024-01-31T22:00:00,2024-01-31T23:00:00,CA,"Føde-, drikke- og tobaksvareindustri",C,Industri,233.154261
3,2024-01-31T22:00:00,2024-01-31T23:00:00,CC,"Træ- og papirindustri, trykkerier",C,Industri,49.051218
4,2024-01-31T22:00:00,2024-01-31T23:00:00,CE_CF_CD,"Kemiskindustri, Olieraffinaderier og Medicinal...",C,Industri,179.019629


In [36]:
from pathlib import Path
import pandas as pd

# parse timestamps
df_month["TimeUTC"] = pd.to_datetime(df_month["TimeUTC"], utc=True)
df_month["TimeDK"]  = pd.to_datetime(df_month["TimeDK"])

df_month = df_month.sort_values(["TimeUTC", "DK19Code"]).reset_index(drop=True)

# save
out_dir = Path("raw/consumption_industry_hourly_monthly")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "consumption_industry_hour_2024_01.parquet"
df_month.to_parquet(out_path, index=False)

print("Saved:", out_path)
print("Rows:", len(df_month))
display(df_month.head())

Saved: data/consumption_industry_hourly_monthly/consumption_industry_hour_2024_01.parquet
Rows: 24552


,TimeUTC,TimeDK,DK36Code,DK36Title,DK19Code,DK19Title,Consumption_MWh
0,2023-12-31 23:00:00+00:00,2024-01-01,A,"Landbrug, skovbrug og fiskeri",A,"Landbrug, skovbrug og fiskeri",174.560800
1,2023-12-31 23:00:00+00:00,2024-01-01,B_E,Råstofindvinding & Vandforsyning og renovation,B_E,Råstofindvinding & Vandforsyning og renovation,153.698786
2,2023-12-31 23:00:00+00:00,2024-01-01,CA,"Føde-, drikke- og tobaksvareindustri",C,Industri,123.434418
3,2023-12-31 23:00:00+00:00,2024-01-01,CC,"Træ- og papirindustri, trykkerier",C,Industri,12.359638
4,2023-12-31 23:00:00+00:00,2024-01-01,CE_CF_CD,"Kemiskindustri, Olieraffinaderier og Medicinal...",C,Industri,116.281177


In [37]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.energidataservice.dk/dataset/ConsumptionDK3619IndustryHour"

ind_dir = Path("raw/consumption_industry_hourly_monthly")
ind_dir.mkdir(parents=True, exist_ok=True)

def fetch_industry_month(year: int, month: int):
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)

    params = {
        "start": start.strftime("%Y-%m-%dT%H:%M"),
        "end":   end.strftime("%Y-%m-%dT%H:%M"),
        "limit": 1000000
    }

    r = requests.get(BASE_URL, params=params, timeout=120)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data.get("records", []))
    if df.empty:
        return None

    df["TimeUTC"] = pd.to_datetime(df["TimeUTC"], utc=True)
    df["TimeDK"]  = pd.to_datetime(df["TimeDK"])

    df = df.sort_values(["TimeUTC", "DK19Code"]).reset_index(drop=True)

    path = ind_dir / f"consumption_industry_hour_{year}_{month:02d}.parquet"
    df.to_parquet(path, index=False)

    return path

def fetch_industry_year(year: int):
    saved, skipped = 0, 0
    for m in range(1, 13):
        path = ind_dir / f"consumption_industry_hour_{year}_{m:02d}.parquet"
        if path.exists():
            continue

        out = fetch_industry_month(year, m)
        if out is None:
            skipped += 1
            print(f"Skipped (no data): {year}-{m:02d}")
        else:
            saved += 1

    return saved, skipped

# Download 2021–2025
saved_total, skipped_total = 0, 0
for y in range(2021, 2026):
    s, k = fetch_industry_year(y)
    saved_total += s
    skipped_total += k

print("\nNew monthly files saved:", saved_total)
print("Months skipped:", skipped_total)

Skipped (no data): 2021-01
Skipped (no data): 2021-02
Skipped (no data): 2021-03
Skipped (no data): 2021-04
Skipped (no data): 2021-05
Skipped (no data): 2021-06

New monthly files saved: 0
Months skipped: 6


In [38]:
import pandas as pd
from pathlib import Path

files = sorted(Path("raw/consumption_industry_hourly_monthly").glob("consumption_industry_hour_*.parquet"))

total_rows = 0
tmin, tmax = None, None

for f in files:
    df = pd.read_parquet(f, columns=["TimeUTC"])
    total_rows += len(df)
    mn = df["TimeUTC"].min()
    mx = df["TimeUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("Files:", len(files))
print("Total rows:", total_rows)
print("Start TimeUTC:", tmin)
print("End TimeUTC:", tmax)

Files: 54
Total rows: 1301745
Start TimeUTC: 2021-06-30 22:00:00+00:00
End TimeUTC: 2025-12-31 22:00:00+00:00


In [40]:
from pathlib import Path
import pandas as pd

p = Path("raw/consumption_industry_hourly_monthly/consumption_industry_hour_2024_01.parquet")

df = pd.read_parquet(p)
print(df.shape)
print(df.dtypes)
df.head()

(24552, 7)
TimeUTC            datetime64[ns, UTC]
TimeDK                  datetime64[ns]
DK36Code                        object
DK36Title                       object
DK19Code                        object
DK19Title                       object
Consumption_MWh                float64
dtype: object


,TimeUTC,TimeDK,DK36Code,DK36Title,DK19Code,DK19Title,Consumption_MWh
0,2023-12-31 23:00:00+00:00,2024-01-01,A,"Landbrug, skovbrug og fiskeri",A,"Landbrug, skovbrug og fiskeri",174.560800
1,2023-12-31 23:00:00+00:00,2024-01-01,B_E,Råstofindvinding & Vandforsyning og renovation,B_E,Råstofindvinding & Vandforsyning og renovation,153.698786
2,2023-12-31 23:00:00+00:00,2024-01-01,CA,"Føde-, drikke- og tobaksvareindustri",C,Industri,123.434418
3,2023-12-31 23:00:00+00:00,2024-01-01,CC,"Træ- og papirindustri, trykkerier",C,Industri,12.359638
4,2023-12-31 23:00:00+00:00,2024-01-01,CE_CF_CD,"Kemiskindustri, Olieraffinaderier og Medicinal...",C,Industri,116.281177


##### Diagnostics

In [43]:
# DATA MANUAL — ConsumptionDK3619IndustryHour
# Prints: file coverage, columns, dtypes, missingness, uniqueness, and basic sanity checks.

import pandas as pd
from pathlib import Path

DATASET_NAME = "ConsumptionDK3619IndustryHour (Hourly industry consumption, DK19/DK36)"
DATA_DIR = Path("raw/consumption_industry_hourly_monthly")
PATTERN = "consumption_industry_hour_*.parquet"

files = sorted(DATA_DIR.glob(PATTERN))
assert files, f"No files found in {DATA_DIR} matching {PATTERN}"

# --- Load one sample file for schema checks ---
sample_file = files[0]
df_sample = pd.read_parquet(sample_file)

print("="*80)
print("DATASET:", DATASET_NAME)
print("DIRECTORY:", DATA_DIR)
print("FILES:", len(files))
print("SAMPLE FILE:", sample_file.name)
print("="*80)

# --- Schema ---
print("\nCOLUMNS:")
for c in df_sample.columns:
    print(" -", c)

print("\nDTYPES (sample file):")
print(df_sample.dtypes)

# --- Coverage across all files (min/max time, total rows) ---
total_rows = 0
tmin, tmax = None, None

for f in files:
    df_t = pd.read_parquet(f, columns=["TimeUTC"])
    total_rows += len(df_t)
    mn = df_t["TimeUTC"].min()
    mx = df_t["TimeUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("\nCOVERAGE:")
print(" - Total rows (all files):", total_rows)
print(" - Start TimeUTC:", tmin)
print(" - End TimeUTC:", tmax)

# --- Key integrity checks on sample file ---
print("\nINTEGRITY CHECKS (sample file):")

key_cols = ["TimeUTC", "DK36Code"]
dup_keys = df_sample.duplicated(key_cols).sum()
print(" - Duplicate (TimeUTC, DK36Code) rows:", int(dup_keys))

# Industry codes present
for col in ["DK36Code", "DK19Code"]:
    if col in df_sample.columns:
        vals = df_sample[col].dropna().unique()
        print(f" - {col} unique (sample): {len(vals)}")
        print("   example:", list(vals[:15]))

# Missingness
print("\nMISSINGNESS (sample file):")
na_counts = df_sample.isna().sum()
for c in df_sample.columns:
    print(f" - {c}: {int(na_counts[c])} missing")

# Consumption sanity checks
cons_col = "Consumption_MWh"
if cons_col in df_sample.columns:
    s = df_sample[cons_col]
    print(f"\nBASIC STATS ({cons_col}) (sample file):")
    print(" - min:", float(s.min()))
    print(" - max:", float(s.max()))
    print(" - mean:", float(s.mean()))
    print(" - zeros:", int((s == 0).sum()))
    print(" - negatives:", int((s < 0).sum()))
    print(" - missing:", int(s.isna().sum()))

# Rows per hour sanity check (sample file)
rows_per_hour = df_sample.groupby("TimeUTC").size()
print("\nROWS PER HOUR (sample file):")
print(" - min:", int(rows_per_hour.min()))
print(" - max:", int(rows_per_hour.max()))
print(" - mode (most common):")
display(rows_per_hour.value_counts().head(10))

# --- Show a snippet ---
print("\nSNIPPET (first 10 rows of sample file):")
display(df_sample.head(10))

DATASET: ConsumptionDK3619IndustryHour (Hourly industry consumption, DK19/DK36)
DIRECTORY: data/consumption_industry_hourly_monthly
FILES: 54
SAMPLE FILE: consumption_industry_hour_2021_07.parquet

COLUMNS:
 - TimeUTC
 - TimeDK
 - DK36Code
 - DK36Title
 - DK19Code
 - DK19Title
 - Consumption_MWh

DTYPES (sample file):
TimeUTC            datetime64[ns, UTC]
TimeDK                  datetime64[ns]
DK36Code                        object
DK36Title                       object
DK19Code                        object
DK19Title                       object
Consumption_MWh                float64
dtype: object

COVERAGE:
 - Total rows (all files): 1301745
 - Start TimeUTC: 2021-06-30 22:00:00+00:00
 - End TimeUTC: 2025-12-31 22:00:00+00:00

INTEGRITY CHECKS (sample file):
 - Duplicate (TimeUTC, DK36Code) rows: 0
 - DK36Code unique (sample): 33
   example: ['A', 'B_E', 'CA', 'CC', 'CE_CF_CD', 'CG', 'CH', 'CI', 'CJ', 'CK', 'CL', 'CM_CB', 'D', 'F', 'G']
 - DK19Code unique (sample): 20
   example: ['

33    744
Name: count, dtype: int64


SNIPPET (first 10 rows of sample file):


,TimeUTC,TimeDK,DK36Code,DK36Title,DK19Code,DK19Title,Consumption_MWh
0,2021-06-30 22:00:00+00:00,2021-07-01,A,"Landbrug, skovbrug og fiskeri",A,"Landbrug, skovbrug og fiskeri",227.503441
1,2021-06-30 22:00:00+00:00,2021-07-01,B_E,Råstofindvinding & Vandforsyning og renovation,B_E,Råstofindvinding & Vandforsyning og renovation,80.454922
2,2021-06-30 22:00:00+00:00,2021-07-01,CA,"Føde-, drikke- og tobaksvareindustri",C,Industri,215.305712
3,2021-06-30 22:00:00+00:00,2021-07-01,CC,"Træ- og papirindustri, trykkerier",C,Industri,47.488769
4,2021-06-30 22:00:00+00:00,2021-07-01,CE_CF_CD,"Kemiskindustri, Olieraffinaderier og Medicinal...",C,Industri,172.549757
5,2021-06-30 22:00:00+00:00,2021-07-01,CG,"Plast-, glas- og betonindustri",C,Industri,144.554341
6,2021-06-30 22:00:00+00:00,2021-07-01,CH,Metalindustri,C,Industri,105.699276
7,2021-06-30 22:00:00+00:00,2021-07-01,CI,Elektronikindustri,C,Industri,7.780121
8,2021-06-30 22:00:00+00:00,2021-07-01,CJ,Fremst. af elektrisk udstyr,C,Industri,12.298962
9,2021-06-30 22:00:00+00:00,2021-07-01,CK,Maskinindustri,C,Industri,45.879539


### API FETCH: Consumption pr grid area

In [44]:
import requests
import pandas as pd

URL = "https://api.energidataservice.dk/dataset/ConsumptionGridAreaHour"

r = requests.get(URL, params={"limit": 50}, timeout=30)
r.raise_for_status()

data = r.json()
df = pd.DataFrame(data.get("records", []))

print("Rows returned:", len(df))
print("Columns:", list(df.columns))
display(df.head())

Rows returned: 50
Columns: ['Date', 'HourBlock6', 'TimeUTC', 'TimeDK', 'PriceArea', 'GridArea', 'GridCompanyName', 'ConsumptionkWh']


,Date,HourBlock6,TimeUTC,TimeDK,PriceArea,GridArea,GridCompanyName,ConsumptionkWh
0,2026-02-11T00:00:00,2026.02.11 00-05,2026-02-11T04:00:00,2026-02-11T05:00:00,DK1,016,El-net Kongerslev A/S,504.620
1,2026-02-11T00:00:00,2026.02.11 00-05,2026-02-11T04:00:00,2026-02-11T05:00:00,DK1,031,Nord Energi Net A/S,139505.845
2,2026-02-11T00:00:00,2026.02.11 00-05,2026-02-11T04:00:00,2026-02-11T05:00:00,DK1,051,Elinord A/S,25371.110
3,2026-02-11T00:00:00,2026.02.11 00-05,2026-02-11T04:00:00,2026-02-11T05:00:00,DK1,084,Kimbrer Elnet A/S,2964.076
4,2026-02-11T00:00:00,2026.02.11 00-05,2026-02-11T04:00:00,2026-02-11T05:00:00,DK1,085,Læsø Elnet A/S,2554.660


In [45]:
import requests
import pandas as pd
from pathlib import Path

URL = "https://api.energidataservice.dk/dataset/ConsumptionGridAreaHour"

params = {
    "start": "2025-01-01T00:00",
    "end":   "2025-02-01T00:00",
    "limit": 1000000
}

r = requests.get(URL, params=params, timeout=120)
r.raise_for_status()
data = r.json()

df_month = pd.DataFrame(data.get("records", []))

print("Rows:", len(df_month))
print("Columns:", list(df_month.columns))
display(df_month.head())

# parse times
df_month["TimeUTC"] = pd.to_datetime(df_month["TimeUTC"], utc=True)
df_month["TimeDK"]  = pd.to_datetime(df_month["TimeDK"])
df_month["Date"]    = pd.to_datetime(df_month["Date"])

df_month = df_month.sort_values(["TimeUTC", "PriceArea", "GridArea"]).reset_index(drop=True)

# save
out_dir = Path("raw/consumption_gridarea_6h_monthly")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "consumption_gridarea_6h_2025_01.parquet"
df_month.to_parquet(out_path, index=False)

print("Saved:", out_path)

Rows: 23064
Columns: ['Date', 'HourBlock6', 'TimeUTC', 'TimeDK', 'PriceArea', 'GridArea', 'GridCompanyName', 'ConsumptionkWh']


,Date,HourBlock6,TimeUTC,TimeDK,PriceArea,GridArea,GridCompanyName,ConsumptionkWh
0,2025-01-31T00:00:00,2025.01.31 00-05,2025-01-31T04:00:00,2025-01-31T05:00:00,DK1,016,El-net Kongerslev A/S,550.530
1,2025-01-31T00:00:00,2025.01.31 00-05,2025-01-31T04:00:00,2025-01-31T05:00:00,DK1,031,Nord Energi Net A/S,132974.299
2,2025-01-31T00:00:00,2025.01.31 00-05,2025-01-31T04:00:00,2025-01-31T05:00:00,DK1,051,Elinord A/S,37853.759
3,2025-01-31T00:00:00,2025.01.31 00-05,2025-01-31T04:00:00,2025-01-31T05:00:00,DK1,084,Kimbrer Elnet A/S,2886.089
4,2025-01-31T00:00:00,2025.01.31 00-05,2025-01-31T04:00:00,2025-01-31T05:00:00,DK1,085,Læsø Elnet A/S,1691.700


Saved: data/consumption_gridarea_6h_monthly/consumption_gridarea_6h_2025_01.parquet


In [46]:
print(df_month["HourBlock6"].unique())
print("Unique TimeUTC per day:", df_month["TimeUTC"].dt.hour.unique())

['2025.01.01 00-05' '2025.01.01 06-11' '2025.01.01 12-17'
 '2025.01.01 18-23' '2025.01.02 00-05' '2025.01.02 06-11'
 '2025.01.02 12-17' '2025.01.02 18-23' '2025.01.03 00-05'
 '2025.01.03 06-11' '2025.01.03 12-17' '2025.01.03 18-23'
 '2025.01.04 00-05' '2025.01.04 06-11' '2025.01.04 12-17'
 '2025.01.04 18-23' '2025.01.05 00-05' '2025.01.05 06-11'
 '2025.01.05 12-17' '2025.01.05 18-23' '2025.01.06 00-05'
 '2025.01.06 06-11' '2025.01.06 12-17' '2025.01.06 18-23'
 '2025.01.07 00-05' '2025.01.07 06-11' '2025.01.07 12-17'
 '2025.01.07 18-23' '2025.01.08 00-05' '2025.01.08 06-11'
 '2025.01.08 12-17' '2025.01.08 18-23' '2025.01.09 00-05'
 '2025.01.09 06-11' '2025.01.09 12-17' '2025.01.09 18-23'
 '2025.01.10 00-05' '2025.01.10 06-11' '2025.01.10 12-17'
 '2025.01.10 18-23' '2025.01.11 00-05' '2025.01.11 06-11'
 '2025.01.11 12-17' '2025.01.11 18-23' '2025.01.12 00-05'
 '2025.01.12 06-11' '2025.01.12 12-17' '2025.01.12 18-23'
 '2025.01.13 00-05' '2025.01.13 06-11' '2025.01.13 12-17'
 '2025.01.13 1

In [47]:
# how many unique TimeUTC per calendar day?
per_day = df_month.groupby(df_month["TimeUTC"].dt.date)["TimeUTC"].nunique()

print("Min per day:", per_day.min())
print("Max per day:", per_day.max())
per_day.head()

Min per day: 1
Max per day: 24


TimeUTC
2024-12-31     1
2025-01-01    24
2025-01-02    24
2025-01-03    24
2025-01-04    24
Name: TimeUTC, dtype: int64

In [48]:
sorted(df_month["TimeUTC"].dt.hour.unique())

[np.int32(0),
 np.int32(1),
 np.int32(2),
 np.int32(3),
 np.int32(4),
 np.int32(5),
 np.int32(6),
 np.int32(7),
 np.int32(8),
 np.int32(9),
 np.int32(10),
 np.int32(11),
 np.int32(12),
 np.int32(13),
 np.int32(14),
 np.int32(15),
 np.int32(16),
 np.int32(17),
 np.int32(18),
 np.int32(19),
 np.int32(20),
 np.int32(21),
 np.int32(22),
 np.int32(23)]

In [49]:
sample = df_month[
    (df_month["GridArea"] == df_month["GridArea"].iloc[0]) &
    (df_month["TimeUTC"].dt.date == df_month["TimeUTC"].dt.date.iloc[0])
]

sample[["HourBlock6","TimeUTC","ConsumptionkWh"]]

,HourBlock6,TimeUTC,ConsumptionkWh
0,2025.01.01 00-05,2024-12-31 23:00:00+00:00,502.23


In [50]:
df_check = df_month[df_month["HourBlock6"] == "2025.01.01 00-05"]

print("Unique TimeUTC in this block:", df_check["TimeUTC"].nunique())
print(sorted(df_check["TimeUTC"].dt.hour.unique()))

Unique TimeUTC in this block: 6
[np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(23)]


In [51]:
df_zonal = (
    df_month
    .groupby(["TimeUTC", "PriceArea"])["ConsumptionkWh"]
    .sum()
    .reset_index()
)

print(df_zonal.head())
print("Rows:", len(df_zonal))

                    TimeUTC PriceArea  ConsumptionkWh
0 2024-12-31 23:00:00+00:00       DK1     2738464.736
1 2024-12-31 23:00:00+00:00       DK2     1523729.309
2 2025-01-01 00:00:00+00:00       DK1     2690947.907
3 2025-01-01 00:00:00+00:00       DK2     1524592.391
4 2025-01-01 01:00:00+00:00       DK1     2741693.228
Rows: 1488


In [52]:
# Load price January 2025
price = pd.read_parquet("raw/elspotprices_monthly/elspot_2025_01.parquet")

# Merge
panel_jan = df_zonal.merge(
    price[["HourUTC", "PriceArea", "SpotPriceDKK"]],
    left_on=["TimeUTC", "PriceArea"],
    right_on=["HourUTC", "PriceArea"],
    how="inner"
)

panel_jan = panel_jan.drop(columns=["HourUTC"])

print("Rows after merge:", len(panel_jan))
panel_jan.head()

Rows after merge: 1488


,TimeUTC,PriceArea,ConsumptionkWh,SpotPriceDKK
0,2024-12-31 23:00:00+00:00,DK1,2738464.736,16.110001
1,2024-12-31 23:00:00+00:00,DK2,1523729.309,18.940001
2,2025-01-01 00:00:00+00:00,DK1,2690947.907,11.930000
3,2025-01-01 00:00:00+00:00,DK2,1524592.391,14.390000
4,2025-01-01 01:00:00+00:00,DK1,2741693.228,12.160000


In [53]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.energidataservice.dk/dataset/ConsumptionGridAreaHour"

grid_dir = Path("raw/consumption_gridarea_hourly_monthly")
grid_dir.mkdir(parents=True, exist_ok=True)

def fetch_grid_month(year: int, month: int):
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)

    params = {
        "start": start.strftime("%Y-%m-%dT%H:%M"),
        "end":   end.strftime("%Y-%m-%dT%H:%M"),
        "limit": 2000000
    }

    r = requests.get(BASE_URL, params=params, timeout=180)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data.get("records", []))
    if df.empty:
        return None

    df["TimeUTC"] = pd.to_datetime(df["TimeUTC"], utc=True)
    df["TimeDK"]  = pd.to_datetime(df["TimeDK"])
    df["Date"]    = pd.to_datetime(df["Date"])

    df = df.sort_values(["TimeUTC", "PriceArea", "GridArea"]).reset_index(drop=True)

    path = grid_dir / f"consumption_gridarea_hour_{year}_{month:02d}.parquet"
    df.to_parquet(path, index=False)

    return path


def fetch_grid_year(year: int):
    saved, skipped = 0, 0

    for m in range(1, 13):
        path = grid_dir / f"consumption_gridarea_hour_{year}_{m:02d}.parquet"
        if path.exists():
            continue

        out = fetch_grid_month(year, m)

        if out is None:
            skipped += 1
            print(f"Skipped (no data): {year}-{m:02d}")
        else:
            saved += 1

    return saved, skipped

In [54]:
saved_total, skipped_total = 0, 0

for y in range(2021, 2026):
    s, k = fetch_grid_year(y)
    saved_total += s
    skipped_total += k

print("\nNew monthly files saved:", saved_total)
print("Months skipped:", skipped_total)

Skipped (no data): 2021-01
Skipped (no data): 2021-02
Skipped (no data): 2021-03
Skipped (no data): 2021-04
Skipped (no data): 2021-05
Skipped (no data): 2021-06

New monthly files saved: 0
Months skipped: 6


In [55]:
import pandas as pd
from pathlib import Path

files = sorted(Path("raw/consumption_gridarea_hourly_monthly").glob("consumption_gridarea_hour_*.parquet"))

total_rows = 0
tmin, tmax = None, None

for f in files:
    df = pd.read_parquet(f, columns=["TimeUTC"])
    total_rows += len(df)

    mn = df["TimeUTC"].min()
    mx = df["TimeUTC"].max()

    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("Files:", len(files))
print("Total rows:", total_rows)
print("Start TimeUTC:", tmin)
print("End TimeUTC:", tmax)

Files: 54
Total rows: 1132700
Start TimeUTC: 2021-06-30 22:00:00+00:00
End TimeUTC: 2025-12-31 22:00:00+00:00


In [56]:
import pandas as pd
from pathlib import Path

sample_year = 2022
files = sorted(Path("raw/consumption_gridarea_hourly_monthly").glob(f"consumption_gridarea_hour_{sample_year}_*.parquet"))

df_y = pd.concat([pd.read_parquet(f, columns=["TimeUTC","PriceArea","GridArea"]) for f in files], ignore_index=True)

print("Unique hours:", df_y["TimeUTC"].nunique())
print("Unique zones:", sorted(df_y["PriceArea"].unique()))
ga_per_hour = df_y.groupby("TimeUTC")["GridArea"].nunique()
print("Grid areas per hour (min/max):", ga_per_hour.min(), ga_per_hour.max())

Unique hours: 8759
Unique zones: ['DK1', 'DK2']
Grid areas per hour (min/max): 7 34


##### Diagnostics

In [57]:
# DATA MANUAL — ConsumptionGridAreaHour (Hourly consumption per grid area)
# Prints: file coverage, columns, dtypes, missingness, uniqueness, and basic sanity checks.

import pandas as pd
from pathlib import Path

DATASET_NAME = "ConsumptionGridAreaHour (Hourly consumption per grid area, includes PriceArea)"
DATA_DIR = Path("raw/consumption_gridarea_hourly_monthly")
PATTERN = "consumption_gridarea_hour_*.parquet"

files = sorted(DATA_DIR.glob(PATTERN))
assert files, f"No files found in {DATA_DIR} matching {PATTERN}"

# --- Load one sample file for schema checks ---
sample_file = files[0]
df_sample = pd.read_parquet(sample_file)

print("="*80)
print("DATASET:", DATASET_NAME)
print("DIRECTORY:", DATA_DIR)
print("FILES:", len(files))
print("SAMPLE FILE:", sample_file.name)
print("="*80)

# --- Schema ---
print("\nCOLUMNS:")
for c in df_sample.columns:
    print(" -", c)

print("\nDTYPES (sample file):")
print(df_sample.dtypes)

# --- Coverage across all files (min/max time, total rows) ---
total_rows = 0
tmin, tmax = None, None

for f in files:
    df_t = pd.read_parquet(f, columns=["TimeUTC"])
    total_rows += len(df_t)
    mn = df_t["TimeUTC"].min()
    mx = df_t["TimeUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("\nCOVERAGE:")
print(" - Total rows (all files):", total_rows)
print(" - Start TimeUTC:", tmin)
print(" - End TimeUTC:", tmax)

# --- Key integrity checks on sample file ---
print("\nINTEGRITY CHECKS (sample file):")

key_cols = ["TimeUTC", "GridArea"]
dup_keys = df_sample.duplicated(key_cols).sum()
print(" - Duplicate (TimeUTC, GridArea) rows:", int(dup_keys))

# Zones and grid areas
if "PriceArea" in df_sample.columns:
    print(" - PriceArea unique values:", sorted(df_sample["PriceArea"].dropna().unique()))

if "GridArea" in df_sample.columns:
    ga = df_sample["GridArea"].dropna().unique()
    print(" - GridArea unique (sample):", len(ga))
    print("   example:", list(ga[:20]))

if "GridCompanyName" in df_sample.columns:
    gc = df_sample["GridCompanyName"].dropna().unique()
    print(" - GridCompanyName unique (sample):", len(gc))
    print("   example:", list(gc[:10]))

# Missingness
print("\nMISSINGNESS (sample file):")
na_counts = df_sample.isna().sum()
for c in df_sample.columns:
    print(f" - {c}: {int(na_counts[c])} missing")

# Consumption sanity checks
cons_col = "ConsumptionkWh"
if cons_col in df_sample.columns:
    s = df_sample[cons_col]
    print(f"\nBASIC STATS ({cons_col}) (sample file):")
    print(" - min:", float(s.min()))
    print(" - max:", float(s.max()))
    print(" - mean:", float(s.mean()))
    print(" - zeros:", int((s == 0).sum()))
    print(" - negatives:", int((s < 0).sum()))
    print(" - missing:", int(s.isna().sum()))

# Grid areas per hour (sample)
ga_per_hour = df_sample.groupby("TimeUTC")["GridArea"].nunique()
print("\nGRID AREAS PER HOUR (sample file):")
print(" - min:", int(ga_per_hour.min()))
print(" - max:", int(ga_per_hour.max()))
print(" - mode (most common):")
display(ga_per_hour.value_counts().head(10))

# Hours per day sanity check (sample file)
hours_per_day = df_sample.groupby(df_sample["TimeUTC"].dt.date)["TimeUTC"].nunique()
print("\nHOURS PER DAY (sample file):")
print(" - min:", int(hours_per_day.min()))
print(" - max:", int(hours_per_day.max()))
display(hours_per_day.value_counts().head(10))

# --- Show a snippet ---
print("\nSNIPPET (first 10 rows of sample file):")
display(df_sample.head(10))

DATASET: ConsumptionGridAreaHour (Hourly consumption per grid area, includes PriceArea)
DIRECTORY: data/consumption_gridarea_hourly_monthly
FILES: 54
SAMPLE FILE: consumption_gridarea_hour_2021_07.parquet

COLUMNS:
 - Date
 - HourBlock6
 - TimeUTC
 - TimeDK
 - PriceArea
 - GridArea
 - GridCompanyName
 - ConsumptionkWh

DTYPES (sample file):
Date                    datetime64[ns]
HourBlock6                      object
TimeUTC            datetime64[ns, UTC]
TimeDK                  datetime64[ns]
PriceArea                       object
GridArea                        object
GridCompanyName                 object
ConsumptionkWh                 float64
dtype: object

COVERAGE:
 - Total rows (all files): 1132700
 - Start TimeUTC: 2021-06-30 22:00:00+00:00
 - End TimeUTC: 2025-12-31 22:00:00+00:00

INTEGRITY CHECKS (sample file):
 - Duplicate (TimeUTC, GridArea) rows: 0
 - PriceArea unique values: ['DK2']
 - GridArea unique (sample): 7
   example: ['755', '757', '791', '853, 740', '854', '860'

GridArea
7    744
Name: count, dtype: int64


HOURS PER DAY (sample file):
 - min: 2
 - max: 24


TimeUTC
24    30
2      1
22     1
Name: count, dtype: int64


SNIPPET (first 10 rows of sample file):


,Date,HourBlock6,TimeUTC,TimeDK,PriceArea,GridArea,GridCompanyName,ConsumptionkWh
0,2021-07-01,2021.07.01 00-05,2021-06-30 22:00:00+00:00,2021-07-01 00:00:00,DK2,755,Radius Elnet A/S (Hillerød),8988.182
1,2021-07-01,2021.07.01 00-05,2021-06-30 22:00:00+00:00,2021-07-01 00:00:00,DK2,757,Elektrus A/S,9054.470
2,2021-07-01,2021.07.01 00-05,2021-06-30 22:00:00+00:00,2021-07-01 00:00:00,DK2,791,Radius Elnet A/S,683073.405
3,2021-07-01,2021.07.01 00-05,2021-06-30 22:00:00+00:00,2021-07-01 00:00:00,DK2,"853, 740","Cerius A/S - 853, Cerius A/S - 740",344604.077
4,2021-07-01,2021.07.01 00-05,2021-06-30 22:00:00+00:00,2021-07-01 00:00:00,DK2,854,NKE-Elnet A/S,7692.840
5,2021-07-01,2021.07.01 00-05,2021-06-30 22:00:00+00:00,2021-07-01 00:00:00,DK2,860,Zeanet A/S,13840.632
6,2021-07-01,2021.07.01 00-05,2021-06-30 22:00:00+00:00,2021-07-01 00:00:00,DK2,911,TREFOR El-Net Øst A/S,17368.930
7,2021-07-01,2021.07.01 00-05,2021-06-30 23:00:00+00:00,2021-07-01 01:00:00,DK2,755,Radius Elnet A/S (Hillerød),8651.573
8,2021-07-01,2021.07.01 00-05,2021-06-30 23:00:00+00:00,2021-07-01 01:00:00,DK2,757,Elektrus A/S,8399.830
9,2021-07-01,2021.07.01 00-05,2021-06-30 23:00:00+00:00,2021-07-01 01:00:00,DK2,791,Radius Elnet A/S,645937.748


### API FETCH Industrial Consumption annually across industry codes data fetch.

In [58]:
import requests
import pandas as pd

URL = "https://api.energidataservice.dk/dataset/ConsumptionDK10IndustryYear"

r = requests.get(URL, params={"limit": 20}, timeout=30)
print("HTTP:", r.status_code)

data = r.json()
df = pd.DataFrame(data.get("records", []))

print("Columns:", list(df.columns))
print("Years available:", sorted(df["Year"].unique()))
df.head()

HTTP: 200
Columns: ['Year', 'RegionName', 'DK10Code', 'DK10Title', 'ConsumptionkWh']
Years available: ['2025-01-01T00:00:00']


,Year,RegionName,DK10Code,DK10Title,ConsumptionkWh
0,2025-01-01T00:00:00,Region Hovedstaden,0.0,Privat,2.503974e+09
1,2025-01-01T00:00:00,Region Hovedstaden,1.0,"Landbrug, skovbrug og fiskeri",4.471476e+07
2,2025-01-01T00:00:00,Region Hovedstaden,2.0,"Industri, råstofindvinding og forsyningsvirkso...",1.762487e+09
3,2025-01-01T00:00:00,Region Hovedstaden,3.0,Bygge og anlæg,1.008204e+08
4,2025-01-01T00:00:00,Region Hovedstaden,4.0,Handel og transport,1.536731e+09


In [59]:
import requests
import pandas as pd

URL = "https://api.energidataservice.dk/dataset/ConsumptionDK10IndustryYear"

r = requests.get(URL, params={"limit": 2000}, timeout=60)
r.raise_for_status()
data = r.json()

df = pd.DataFrame(data["records"])
print("Unique Year values (sample):", sorted(df["Year"].unique())[:10])
print("Max Year:", max(df["Year"]))
print("Min Year:", min(df["Year"]))


Unique Year values (sample): ['2021-01-01T00:00:00', '2022-01-01T00:00:00', '2023-01-01T00:00:00', '2024-01-01T00:00:00', '2025-01-01T00:00:00']
Max Year: 2025-01-01T00:00:00
Min Year: 2021-01-01T00:00:00


In [60]:
from pathlib import Path
import requests
import pandas as pd

BASE_URL = "https://api.energidataservice.dk/dataset/ConsumptionDK10IndustryYear"

dk10_dir = Path("raw/consumption_dk10_region_year")
dk10_dir.mkdir(parents=True, exist_ok=True)

def fetch_year_window(year: int):
    params = {
        "start": f"{year}-01-01T00:00",
        "end":   f"{year+1}-01-01T00:00",
        "limit": 100000
    }
    r = requests.get(BASE_URL, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data.get("records", []))
    if df.empty:
        print(f"Skipped (no data): {year}")
        return None

    # Parse year column to datetime (nice to have)
    df["Year"] = pd.to_datetime(df["Year"])

    path = dk10_dir / f"consumption_dk10_region_{year}.parquet"
    df.to_parquet(path, index=False)
    print(f"Saved {year}: {len(df)} rows")
    return path

for y in range(2021, 2026):
    fetch_year_window(y)

Saved 2021: 60 rows
Saved 2022: 60 rows
Saved 2023: 60 rows
Saved 2024: 60 rows
Saved 2025: 60 rows


In [61]:
from pathlib import Path
files = sorted(Path("raw/consumption_dk10_region_year").glob("consumption_dk10_region_*.parquet"))
print([f.name for f in files])

['consumption_dk10_region_2021.parquet', 'consumption_dk10_region_2022.parquet', 'consumption_dk10_region_2023.parquet', 'consumption_dk10_region_2024.parquet', 'consumption_dk10_region_2025.parquet']


##### Diagnostics

In [62]:
# DATA MANUAL — ConsumptionDK10IndustryYear (Annual industry consumption by region)
# Prints: file coverage, columns, dtypes, missingness, uniqueness, and basic sanity checks.

import pandas as pd
from pathlib import Path

DATASET_NAME = "ConsumptionDK10IndustryYear (Annual consumption by Region × DK10 industry)"
DATA_DIR = Path("raw/consumption_dk10_region_year")
PATTERN = "consumption_dk10_region_*.parquet"

files = sorted(DATA_DIR.glob(PATTERN))
assert files, f"No files found in {DATA_DIR} matching {PATTERN}"

# --- Load all files (small dataset) ---
df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print("="*80)
print("DATASET:", DATASET_NAME)
print("DIRECTORY:", DATA_DIR)
print("FILES:", len(files))
print("="*80)

# --- Schema ---
print("\nCOLUMNS:")
for c in df.columns:
    print(" -", c)

print("\nDTYPES:")
print(df.dtypes)

# --- Coverage ---
df["Year"] = pd.to_datetime(df["Year"])

print("\nCOVERAGE:")
print(" - Years:", sorted(df["Year"].dt.year.unique()))
print(" - Total rows:", len(df))

# --- Regions and industries ---
if "RegionName" in df.columns:
    regions = df["RegionName"].dropna().unique()
    print("\nREGIONS:")
    print(" - Count:", len(regions))
    print(" - Values:", sorted(regions))

if "DK10Code" in df.columns:
    industries = df["DK10Code"].dropna().unique()
    print("\nDK10 INDUSTRY CODES:")
    print(" - Count:", len(industries))
    print(" - Values:", sorted(industries))

# --- Observations per year ---
print("\nROWS PER YEAR:")
display(df.groupby(df["Year"].dt.year).size())

# --- Observations per region per year ---
print("\nROWS PER YEAR × REGION:")
display(df.groupby([df["Year"].dt.year, "RegionName"]).size())

# --- Missingness ---
print("\nMISSINGNESS:")
na_counts = df.isna().sum()
for c in df.columns:
    print(f" - {c}: {int(na_counts[c])} missing")

# --- Consumption sanity checks ---
cons_col = "ConsumptionkWh"
if cons_col in df.columns:
    s = df[cons_col]
    print(f"\nBASIC STATS ({cons_col}):")
    print(" - min:", float(s.min()))
    print(" - max:", float(s.max()))
    print(" - mean:", float(s.mean()))
    print(" - zeros:", int((s == 0).sum()))
    print(" - negatives:", int((s < 0).sum()))
    print(" - missing:", int(s.isna().sum()))

# --- Snippet ---
print("\nSNIPPET (first 10 rows):")
display(df.head(10))

DATASET: ConsumptionDK10IndustryYear (Annual consumption by Region × DK10 industry)
DIRECTORY: data/consumption_dk10_region_year
FILES: 5

COLUMNS:
 - Year
 - RegionName
 - DK10Code
 - DK10Title
 - ConsumptionkWh

DTYPES:
Year              datetime64[ns]
RegionName                object
DK10Code                 float64
DK10Title                 object
ConsumptionkWh           float64
dtype: object

COVERAGE:
 - Years: [np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025)]
 - Total rows: 300

REGIONS:
 - Count: 5
 - Values: ['Region Hovedstaden', 'Region Midtjylland', 'Region Nordjylland', 'Region Sjælland', 'Region Syddanmark']

DK10 INDUSTRY CODES:
 - Count: 12
 - Values: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(10.0), np.float64(11.0)]

ROWS PER YEAR:


Year
2021    60
2022    60
2023    60
2024    60
2025    60
dtype: int64


ROWS PER YEAR × REGION:


Year  RegionName        
2021  Region Hovedstaden    12
      Region Midtjylland    12
      Region Nordjylland    12
      Region Sjælland       12
      Region Syddanmark     12
2022  Region Hovedstaden    12
      Region Midtjylland    12
      Region Nordjylland    12
      Region Sjælland       12
      Region Syddanmark     12
2023  Region Hovedstaden    12
      Region Midtjylland    12
      Region Nordjylland    12
      Region Sjælland       12
      Region Syddanmark     12
2024  Region Hovedstaden    12
      Region Midtjylland    12
      Region Nordjylland    12
      Region Sjælland       12
      Region Syddanmark     12
2025  Region Hovedstaden    12
      Region Midtjylland    12
      Region Nordjylland    12
      Region Sjælland       12
      Region Syddanmark     12
dtype: int64


MISSINGNESS:
 - Year: 0 missing
 - RegionName: 0 missing
 - DK10Code: 0 missing
 - DK10Title: 0 missing
 - ConsumptionkWh: 0 missing

BASIC STATS (ConsumptionkWh):
 - min: 17503806.771999
 - max: 3579989744.144974
 - mean: 572964121.6778775
 - zeros: 0
 - negatives: 0
 - missing: 0

SNIPPET (first 10 rows):


,Year,RegionName,DK10Code,DK10Title,ConsumptionkWh
0,2021-01-01,Region Hovedstaden,0.0,Privat,2.521477e+09
1,2021-01-01,Region Hovedstaden,1.0,"Landbrug, skovbrug og fiskeri",4.682711e+07
2,2021-01-01,Region Hovedstaden,2.0,"Industri, råstofindvinding og forsyningsvirkso...",1.243942e+09
3,2021-01-01,Region Hovedstaden,3.0,Bygge og anlæg,9.468843e+07
4,2021-01-01,Region Hovedstaden,4.0,Handel og transport,1.339431e+09
5,2021-01-01,Region Hovedstaden,5.0,Information og kommunikation,3.699139e+08
6,2021-01-01,Region Hovedstaden,6.0,Finansiering og forsikring,1.306658e+08
7,2021-01-01,Region Hovedstaden,7.0,Ejendomshandel og udlejning,6.302510e+08
8,2021-01-01,Region Hovedstaden,8.0,Erhvervsservice,3.665415e+08
9,2021-01-01,Region Hovedstaden,9.0,"Offentlig administration, undervisning og sundhed",8.588659e+08


### API FETCH: WIND AND SOLAR DATA

In [63]:
r = requests.get("https://api.energidataservice.dk/dataset/Forecasts_Hour?limit=20")
df = pd.DataFrame(r.json()["records"])
print(df.columns)

Index(['HourUTC', 'HourDK', 'PriceArea', 'ForecastType', 'ForecastDayAhead',
       'ForecastIntraday', 'Forecast5Hour', 'Forecast1Hour', 'ForecastCurrent',
       'TimestampUTC', 'TimestampDK'],
      dtype='object')


In [64]:
df["ForecastType"].unique()

array(['Offshore Wind', 'Onshore Wind', 'Solar'], dtype=object)

In [65]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.energidataservice.dk/dataset/Forecasts_Hour"

forecast_dir = Path("raw/forecast_hourly_monthly")
forecast_dir.mkdir(parents=True, exist_ok=True)

def fetch_forecast_month(year: int, month: int):
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthBegin(1)

    params = {
        "start": start.strftime("%Y-%m-%dT%H:%M"),
        "end":   end.strftime("%Y-%m-%dT%H:%M"),
        "limit": 2000000
    }

    r = requests.get(BASE_URL, params=params, timeout=180)
    r.raise_for_status()
    data = r.json()

    df = pd.DataFrame(data.get("records", []))
    if df.empty:
        return None

    df["HourUTC"] = pd.to_datetime(df["HourUTC"], utc=True)
    df["HourDK"]  = pd.to_datetime(df["HourDK"])
    df["TimestampUTC"] = pd.to_datetime(df["TimestampUTC"], utc=True)
    df["TimestampDK"]  = pd.to_datetime(df["TimestampDK"])

    df = df.sort_values(["HourUTC","PriceArea","ForecastType"]).reset_index(drop=True)

    path = forecast_dir / f"forecast_hour_{year}_{month:02d}.parquet"
    df.to_parquet(path, index=False)

    return path


def fetch_forecast_year(year: int):
    saved, skipped = 0, 0

    for m in range(1, 13):
        path = forecast_dir / f"forecast_hour_{year}_{m:02d}.parquet"
        if path.exists():
            continue

        out = fetch_forecast_month(year, m)

        if out is None:
            skipped += 1
            print(f"Skipped (no data): {year}-{m:02d}")
        else:
            saved += 1

    return saved, skipped

In [66]:
saved_total, skipped_total = 0, 0

for y in range(2021, 2026):
    s, k = fetch_forecast_year(y)
    saved_total += s
    skipped_total += k

print("\nNew monthly files saved:", saved_total)
print("Months skipped:", skipped_total)


New monthly files saved: 0
Months skipped: 0


In [67]:
from pathlib import Path
import pandas as pd

files = sorted(Path("raw/forecast_hourly_monthly").glob("forecast_hour_*.parquet"))

total_rows = 0
tmin, tmax = None, None

for f in files:
    df = pd.read_parquet(f, columns=["HourUTC"])
    total_rows += len(df)
    mn = df["HourUTC"].min()
    mx = df["HourUTC"].max()

    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("Files:", len(files))
print("Total rows:", total_rows)
print("Start HourUTC:", tmin)
print("End HourUTC:", tmax)

Files: 60
Total rows: 262034
Start HourUTC: 2020-12-31 23:00:00+00:00
End HourUTC: 2025-12-31 22:00:00+00:00


In [68]:
import pandas as pd
from pathlib import Path

src_dir = Path("raw/forecast_hourly_monthly")
out_dir = Path("raw/forecast_hourly_monthly_compact")
out_dir.mkdir(parents=True, exist_ok=True)

files = sorted(src_dir.glob("forecast_hour_*.parquet"))

for f in files:
    df = pd.read_parquet(f, columns=["HourUTC","PriceArea","ForecastType","ForecastDayAhead"])
    df["HourUTC"] = pd.to_datetime(df["HourUTC"], utc=True)

    # keep only relevant forecast types
    df = df[df["ForecastType"].isin(["Offshore Wind","Onshore Wind","Solar"])]

    # aggregate: Wind = Offshore + Onshore (per hour, zone)
    wind = (
        df[df["ForecastType"].isin(["Offshore Wind","Onshore Wind"])]
        .groupby(["HourUTC","PriceArea"], as_index=False)["ForecastDayAhead"]
        .sum()
        .rename(columns={"ForecastDayAhead":"Wind_DayAhead"})
    )

    # solar kept separate (per hour, zone)
    solar = (
        df[df["ForecastType"]=="Solar"]
        .groupby(["HourUTC","PriceArea"], as_index=False)["ForecastDayAhead"]
        .sum()
        .rename(columns={"ForecastDayAhead":"Solar_DayAhead"})
    )

    # merge wind + solar
    out = wind.merge(solar, on=["HourUTC","PriceArea"], how="outer").sort_values(["HourUTC","PriceArea"])

    out_path = out_dir / f.name.replace("forecast_hour_", "forecast_compact_")
    out.to_parquet(out_path, index=False)

print("Saved compact forecast files to:", out_dir)

Saved compact forecast files to: data/forecast_hourly_monthly_compact


In [69]:
from pathlib import Path
import pandas as pd

# pick one compact file
files = sorted(Path("raw/forecast_hourly_monthly_compact").glob("forecast_compact_*.parquet"))
print("Example file:", files[0].name)

df = pd.read_parquet(files[0])

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head(30)

Example file: forecast_compact_2021_01.parquet
Rows: 1488
Columns: ['HourUTC', 'PriceArea', 'Wind_DayAhead', 'Solar_DayAhead']


,HourUTC,PriceArea,Wind_DayAhead,Solar_DayAhead
0,2020-12-31 23:00:00+00:00,DK1,344.541664,0.000000
1,2020-12-31 23:00:00+00:00,DK2,313.333328,0.000000
2,2021-01-01 00:00:00+00:00,DK1,220.250008,0.000000
3,2021-01-01 00:00:00+00:00,DK2,278.625000,0.000000
4,2021-01-01 01:00:00+00:00,DK1,170.458336,0.000000
5,2021-01-01 01:00:00+00:00,DK2,230.125000,0.000000
6,2021-01-01 02:00:00+00:00,DK1,131.875004,0.000000
7,2021-01-01 02:00:00+00:00,DK2,196.375000,0.000000
8,2021-01-01 03:00:00+00:00,DK1,116.750000,0.000000
9,2021-01-01 03:00:00+00:00,DK2,163.500000,0.000000


#### Diagnostics

In [ ]:
import pandas as pd
from pathlib import Path

DATASET_NAME = "Forecasts_Hour (compact: Wind_DayAhead + Solar_DayAhead)"
DATA_DIR = Path("raw/forecast_hourly_monthly_compact")
PATTERN = "forecast_compact_*.parquet"

files = sorted(DATA_DIR.glob(PATTERN))
assert files, f"No files found in {DATA_DIR} matching {PATTERN}"

# --- Load one sample file for schema checks ---
sample_file = files[0]
df_sample = pd.read_parquet(sample_file)

print("="*80)
print("DATASET:", DATASET_NAME)
print("DIRECTORY:", DATA_DIR)
print("FILES:", len(files))
print("SAMPLE FILE:", sample_file.name)
print("="*80)

# --- Schema ---
print("\nCOLUMNS:")
for c in df_sample.columns:
    print(" -", c)

print("\nDTYPES (sample file):")
print(df_sample.dtypes)

# --- Coverage across all files (min/max time, total rows) ---
total_rows = 0
tmin, tmax = None, None

for f in files:
    df_t = pd.read_parquet(f, columns=["HourUTC"])
    total_rows += len(df_t)
    mn = df_t["HourUTC"].min()
    mx = df_t["HourUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("\nCOVERAGE:")
print(" - Total rows (all files):", total_rows)
print(" - Start HourUTC:", tmin)
print(" - End HourUTC:", tmax)

# --- Key integrity checks on sample file ---
print("\nINTEGRITY CHECKS (sample file):")

# 1) Key uniqueness
key_cols = ["HourUTC", "PriceArea"]
dup_keys = df_sample.duplicated(key_cols).sum()
print(" - Duplicate (HourUTC, PriceArea) rows:", int(dup_keys))

# 2) PriceArea values
if "PriceArea" in df_sample.columns:
    print(" - PriceArea unique values:", sorted(df_sample["PriceArea"].dropna().unique()))

# 3) Missingness
print("\nMISSINGNESS (sample file):")
na_counts = df_sample.isna().sum()
for c in df_sample.columns:
    print(f" - {c}: {int(na_counts[c])} missing")

# 4) Basic stats for forecast columns
for col in ["Wind_DayAhead", "Solar_DayAhead"]:
    if col in df_sample.columns:
        s = df_sample[col]
        print(f"\nBASIC STATS ({col}) (sample file):")
        print(" - min:", float(s.min()))
        print(" - max:", float(s.max()))
        print(" - mean:", float(s.mean()))
        print(" - zeros:", int((s == 0).sum()))
        print(" - negatives:", int((s < 0).sum()))

# --- Show a snippet ---
print("\nSNIPPET (first 10 rows of sample file):")
display(df_sample.head(10))

DATASET: Forecasts_Hour (compact: Wind_DayAhead + Solar_DayAhead)
DIRECTORY: data/forecast_hourly_monthly_compact
FILES: 60
SAMPLE FILE: forecast_compact_2021_01.parquet

COLUMNS:
 - HourUTC
 - PriceArea
 - Wind_DayAhead
 - Solar_DayAhead

DTYPES (sample file):
HourUTC           datetime64[ns, UTC]
PriceArea                      object
Wind_DayAhead                 float64
Solar_DayAhead                float64
dtype: object

COVERAGE:
 - Total rows (all files): 87404
 - Start HourUTC: 2020-12-31 23:00:00+00:00
 - End HourUTC: 2025-12-31 22:00:00+00:00

INTEGRITY CHECKS (sample file):
 - Duplicate (HourUTC, PriceArea) rows: 0
 - PriceArea unique values: ['DK1', 'DK2']

MISSINGNESS (sample file):
 - HourUTC: 0 missing
 - PriceArea: 0 missing
 - Wind_DayAhead: 0 missing
 - Solar_DayAhead: 0 missing

BASIC STATS (Wind_DayAhead) (sample file):
 - min: 7.0
 - max: 3638.541748
 - mean: 786.3101761518816
 - zeros: 0
 - negatives: 0

BASIC STATS (Solar_DayAhead) (sample file):
 - min: 0.0
 - ma

,HourUTC,PriceArea,Wind_DayAhead,Solar_DayAhead
0,2020-12-31 23:00:00+00:00,DK1,344.541664,0.0
1,2020-12-31 23:00:00+00:00,DK2,313.333328,0.0
2,2021-01-01 00:00:00+00:00,DK1,220.250008,0.0
3,2021-01-01 00:00:00+00:00,DK2,278.625000,0.0
4,2021-01-01 01:00:00+00:00,DK1,170.458336,0.0
5,2021-01-01 01:00:00+00:00,DK2,230.125000,0.0
6,2021-01-01 02:00:00+00:00,DK1,131.875004,0.0
7,2021-01-01 02:00:00+00:00,DK2,196.375000,0.0
8,2021-01-01 03:00:00+00:00,DK1,116.750000,0.0
9,2021-01-01 03:00:00+00:00,DK2,163.500000,0.0


### API Fetch: TSO AND DSO PRICES

In [71]:
import requests
import pandas as pd

URL = "https://api.energidataservice.dk/dataset/DatahubPricelist"

r = requests.get(URL, params={"limit": 20}, timeout=30)
r.raise_for_status()

data = r.json()
df = pd.DataFrame(data["records"])

print("Columns:")
print(list(df.columns))

print("\nSample:")
display(df.head())

Columns:
['ChargeOwner', 'GLN_Number', 'ChargeType', 'ChargeTypeCode', 'Note', 'Description', 'ValidFrom', 'ValidTo', 'VATClass', 'Price1', 'Price2', 'Price3', 'Price4', 'Price5', 'Price6', 'Price7', 'Price8', 'Price9', 'Price10', 'Price11', 'Price12', 'Price13', 'Price14', 'Price15', 'Price16', 'Price17', 'Price18', 'Price19', 'Price20', 'Price21', 'Price22', 'Price23', 'Price24', 'TransparentInvoicing', 'TaxIndicator', 'ResolutionDuration']

Sample:


,ChargeOwner,GLN_Number,ChargeType,ChargeTypeCode,Note,Description,ValidFrom,ValidTo,VATClass,Price1,...,Price18,Price19,Price20,Price21,Price22,Price23,Price24,TransparentInvoicing,TaxIndicator,ResolutionDuration
0,TREFOR El-net A/S,5790000392261,D01,E-150,Prisnedsættelse Net abo C forbrug skabelon,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2018-01-01T00:00:00,2019-01-01T00:00:00,D02,-35.9167,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,P1M
1,TREFOR El-net A/S,5790000392261,D01,E-150,Prisnedsættelse Net abo C forbrug skabelon,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2017-05-01T00:00:00,2018-01-01T00:00:00,D02,-39.5833,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,P1M
2,TREFOR El-net A/S,5790000392261,D01,E-150,Prisnedsættelse Net abo C forbrug skabelon,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2017-01-01T00:00:00,2017-05-01T00:00:00,D02,-39.5800,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,P1M
3,TREFOR El-net A/S,5790000392261,D01,E-150,Prisnedsættelse Net abo C forbrug skabelon,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2016-10-01T00:00:00,2017-01-01T00:00:00,D02,-41.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,P1M
4,TREFOR El-net A/S,5790000392261,D01,E-150,Prisnedsættelse Net abo C forbrug skabelon,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2016-08-05T00:00:00,2016-10-01T00:00:00,D02,-41.0000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M


In [72]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.energidataservice.dk/dataset/DatahubPricelist"

out_dir = Path("raw/datahub_pricelist")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "datahub_pricelist_validfrom_2021_2025.parquet"

params = {
    "start": "2021-01-01T00:00",
    "end":   "2026-01-01T00:00",   # end exclusive
    "limit": 2000000
}

r = requests.get(BASE_URL, params=params, timeout=180)
r.raise_for_status()
data = r.json()

df = pd.DataFrame(data.get("records", []))

print("Rows:", len(df))
print("Columns:", len(df.columns))

# parse dates
df["ValidFrom"] = pd.to_datetime(df["ValidFrom"])
df["ValidTo"]   = pd.to_datetime(df["ValidTo"])

# save
df.to_parquet(out_path, index=False)

print("Saved:", out_path)

Rows: 327517
Columns: 36
Saved: data/datahub_pricelist/datahub_pricelist_validfrom_2021_2025.parquet


In [73]:
print("ValidFrom min/max:", df["ValidFrom"].min(), df["ValidFrom"].max())
print("ValidTo min/max:", df["ValidTo"].min(), df["ValidTo"].max())
print("Unique ChargeOwners:", df["ChargeOwner"].nunique())
print("Unique ChargeTypes:", df["ChargeType"].nunique())
print("Unique ChargeTypeCodes:", df["ChargeTypeCode"].nunique())

ValidFrom min/max: 2021-01-01 00:00:00 2025-12-31 00:00:00
ValidTo min/max: 2021-01-02 00:00:00 2026-04-01 00:00:00
Unique ChargeOwners: 45
Unique ChargeTypes: 3
Unique ChargeTypeCodes: 2744


##### Diagnostics

In [74]:
# DATA MANUAL — DatahubPricelist (2021–2025 subset)
# Prints: coverage, schema, validity structure, hourly price structure, missingness, and sanity checks.

import pandas as pd
from pathlib import Path

DATASET_NAME = "DatahubPricelist (ValidFrom 2021–2025)"
DATA_PATH = Path("raw/datahub_pricelist/datahub_pricelist_validfrom_2021_2025.parquet")

assert DATA_PATH.exists(), f"File not found: {DATA_PATH}"

df = pd.read_parquet(DATA_PATH)

print("="*80)
print("DATASET:", DATASET_NAME)
print("FILE:", DATA_PATH.name)
print("="*80)

# --- Schema ---
print("\nCOLUMNS:")
for c in df.columns:
    print(" -", c)

print("\nDTYPES:")
print(df.dtypes)

# --- Coverage ---
df["ValidFrom"] = pd.to_datetime(df["ValidFrom"])
df["ValidTo"]   = pd.to_datetime(df["ValidTo"])

print("\nCOVERAGE:")
print(" - Rows:", len(df))
print(" - ValidFrom min/max:", df["ValidFrom"].min(), df["ValidFrom"].max())
print(" - ValidTo min/max:", df["ValidTo"].min(), df["ValidTo"].max())

# --- Key dimensions ---
print("\nKEY DIMENSIONS:")
print(" - Unique ChargeOwners:", df["ChargeOwner"].nunique())
print(" - Unique GLN_Number:", df["GLN_Number"].nunique())
print(" - Unique ChargeType:", df["ChargeType"].nunique())
print(" - Unique ChargeTypeCode:", df["ChargeTypeCode"].nunique())
print(" - Unique VATClass:", df["VATClass"].nunique())
print(" - Unique ResolutionDuration:", df["ResolutionDuration"].nunique())

# --- Observations per year ---
print("\nROWS PER VALIDFROM YEAR:")
display(df.groupby(df["ValidFrom"].dt.year).size())

# --- Missingness ---
print("\nMISSINGNESS:")
na_counts = df.isna().sum()
for c in df.columns:
    print(f" - {c}: {int(na_counts[c])} missing")

# --- Hourly price structure ---
price_cols = [c for c in df.columns if c.startswith("Price")]
print("\nHOURLY PRICE COLUMNS:")
print(" - Count:", len(price_cols))
print(" - Example columns:", price_cols[:5], "...", price_cols[-5:])

# Flat rate detection (only Price1 used)
if "Price1" in df.columns:
    flat_rate = (
        df[price_cols].fillna(0)
        .apply(lambda row: (row[1:] == 0).all(), axis=1)
        .sum()
    )
    print("\nFLAT RATE ROWS (only Price1 used):", int(flat_rate))

# Basic stats for Price1
if "Price1" in df.columns:
    s = df["Price1"]
    print("\nBASIC STATS (Price1):")
    print(" - min:", float(s.min()))
    print(" - max:", float(s.max()))
    print(" - mean:", float(s.mean()))
    print(" - zeros:", int((s == 0).sum()))
    print(" - negatives:", int((s < 0).sum()))

# --- Snippet ---
print("\nSNIPPET (first 10 rows):")
display(df.head(10))

DATASET: DatahubPricelist (ValidFrom 2021–2025)
FILE: datahub_pricelist_validfrom_2021_2025.parquet

COLUMNS:
 - ChargeOwner
 - GLN_Number
 - ChargeType
 - ChargeTypeCode
 - Note
 - Description
 - ValidFrom
 - ValidTo
 - VATClass
 - Price1
 - Price2
 - Price3
 - Price4
 - Price5
 - Price6
 - Price7
 - Price8
 - Price9
 - Price10
 - Price11
 - Price12
 - Price13
 - Price14
 - Price15
 - Price16
 - Price17
 - Price18
 - Price19
 - Price20
 - Price21
 - Price22
 - Price23
 - Price24
 - TransparentInvoicing
 - TaxIndicator
 - ResolutionDuration

DTYPES:
ChargeOwner                     object
GLN_Number                      object
ChargeType                      object
ChargeTypeCode                  object
Note                            object
Description                     object
ValidFrom               datetime64[ns]
ValidTo                 datetime64[ns]
VATClass                        object
Price1                         float64
Price2                         float64
Price3         

ValidFrom
2021    31946
2022    54736
2023    79561
2024    86624
2025    74650
dtype: int64


MISSINGNESS:
 - ChargeOwner: 0 missing
 - GLN_Number: 0 missing
 - ChargeType: 0 missing
 - ChargeTypeCode: 0 missing
 - Note: 0 missing
 - Description: 0 missing
 - ValidFrom: 0 missing
 - ValidTo: 894 missing
 - VATClass: 0 missing
 - Price1: 0 missing
 - Price2: 3339 missing
 - Price3: 3339 missing
 - Price4: 3339 missing
 - Price5: 3339 missing
 - Price6: 3339 missing
 - Price7: 3339 missing
 - Price8: 3339 missing
 - Price9: 3339 missing
 - Price10: 3339 missing
 - Price11: 3339 missing
 - Price12: 3339 missing
 - Price13: 3339 missing
 - Price14: 3339 missing
 - Price15: 3339 missing
 - Price16: 3339 missing
 - Price17: 3339 missing
 - Price18: 3339 missing
 - Price19: 3339 missing
 - Price20: 3339 missing
 - Price21: 3339 missing
 - Price22: 3339 missing
 - Price23: 3339 missing
 - Price24: 3339 missing
 - TransparentInvoicing: 0 missing
 - TaxIndicator: 0 missing
 - ResolutionDuration: 0 missing

HOURLY PRICE COLUMNS:
 - Count: 24
 - Example columns: ['Price1', 'Price2', 'Pric

,ChargeOwner,GLN_Number,ChargeType,ChargeTypeCode,Note,Description,ValidFrom,ValidTo,VATClass,Price1,...,Price18,Price19,Price20,Price21,Price22,Price23,Price24,TransparentInvoicing,TaxIndicator,ResolutionDuration
0,TREFOR El-net A/S,5790000392261,D01,E-50,Net abo C forbrug skabelon,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2021-01-01,2022-01-01,D02,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
1,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2025-04-01,2026-01-01,D02,37.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
2,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug time,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2025-01-01,2025-04-01,D02,37.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
3,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug time,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2024-05-31,2025-01-01,D02,40.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
4,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug time,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2024-05-02,2024-05-31,D02,40.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
5,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2024-05-01,2024-05-02,D02,40.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
6,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug time,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2024-01-01,2024-05-01,D02,40.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
7,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug time,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2023-01-01,2024-01-01,D02,34.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
8,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug time,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2022-01-02,2023-01-01,D02,37.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M
9,TREFOR El-net A/S,5790000392261,D01,E-51,Net abo C forbrug time,"Abonnement, hvor aftagepunktet typisk er i 0,4...",2022-01-01,2022-01-02,D02,37.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,P1M


### API FETCH TEMPERATUR DATA: 

#### Decide on weather stations

In [77]:
import requests
import pandas as pd

# Fetch all active stations
url = "https://opendataapi.dmi.dk/v2/metObs/collections/station/items"

params = {
    "limit": 1000,
    "status": "Active",
    "bbox": "7,54,16,58"
}

r = requests.get(url, params=params)
r.raise_for_status()

data = r.json()
stations = pd.json_normalize(data["features"])

# Extract relevant columns
stations = stations[[
    "properties.stationId",
    "properties.name",
    "geometry.coordinates"
]].copy()

stations["lon"] = stations["geometry.coordinates"].apply(lambda x: x[0])
stations["lat"] = stations["geometry.coordinates"].apply(lambda x: x[1])

stations.rename(columns={
    "properties.stationId": "stationId",
    "properties.name": "name"
}, inplace=True)

stations.head()

,stationId,name,geometry.coordinates,lon,lat
0,20085,Lendum,"[10.2705, 57.4029]",10.2705,57.4029
1,20119,Brønderslev,"[9.9592, 57.2825]",9.9592,57.2825
2,20119,Brønderslev,"[9.9592, 57.2825]",9.9592,57.2825
3,06073,Sletterhage Fyr,"[10.5135, 56.0955]",10.5135,56.0955
4,20228,Voerså Hede,"[10.4708, 57.2214]",10.4708,57.2214


In [78]:
stations["zone"] = stations["lon"].apply(lambda x: "DK1" if x < 10.8 else "DK2")

print(stations["zone"].value_counts())

zone
DK1    334
DK2    172
Name: count, dtype: int64


In [96]:
import numpy as np
import pandas as pd

TEMP_PARAM = "temp_dry"
LON_SPLIT = 10.8  # DK1/DK2 divider

def select_spread_temp_stations(stations_df, zone, n=10, random_state=42):
    df = stations_df.copy()

    # ensure zone exists
    df["zone"] = df["lon"].apply(lambda x: "DK1" if x < LON_SPLIT else "DK2")

    # keep only stations that support temp_dry
    df = df[df["parameterId"].apply(lambda xs: isinstance(xs, list) and (TEMP_PARAM in xs))].copy()

    # filter zone
    df = df[df["zone"] == zone].copy()

    # drop duplicates
    df = df.drop_duplicates(subset=["stationId"])

    # avoid lighthouse/offshore-ish names + Bornholm
    df = df[~df["name"].str.lower().str.contains("fyr|fyrskib|rev", na=False)]
    bornholm = (df["lon"] > 14.0) & (df["lat"] > 54.9) & (df["lat"] < 55.4)
    df = df[~bornholm]

    # spread by latitude bins
    df = df.sort_values("lat").reset_index(drop=True)
    if len(df) < n:
        raise ValueError(f"Not enough temp stations in {zone}: {len(df)} < {n}")

    df["lat_bin"] = pd.qcut(df["lat"], q=n, duplicates="drop")
    sel = (
        df.groupby("lat_bin", group_keys=False)
          .apply(lambda x: x.sample(1, random_state=random_state))
          .reset_index(drop=True)
          .sort_values("lat")
          .reset_index(drop=True)
    )

    return sel[["stationId","name","lat","lon"]]

dk1_selected = select_spread_temp_stations(stations, "DK1", n=10)
dk2_selected = select_spread_temp_stations(stations, "DK2", n=10)

display(dk1_selected)
display(dk2_selected)

/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/691213029.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("lat_bin", group_keys=False)
/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/691213029.py:35: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(1, random_state=random_state))
/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/691213029.py:34: FutureWarning: The default of observed=False is deprecated and will be changed to T

,stationId,name,lat,lon
0,06118,Sønderborg Lufthavn,54.9616,9.7930
1,06093,Vester Vedsted,55.2908,8.6551
2,06088,Nordby,55.4483,8.4003
3,06104,Billund Lufthavn,55.7379,9.1674
4,06102,HORSENS/BYGHOLM,55.8680,9.7872
5,06039,Galten,56.1618,9.9033
6,06072,Ødum,56.3027,10.1272
7,06049,Hald Vest,56.5604,10.0929
8,06051,Vestervig,56.7637,8.3207
9,06032,Stenhøj,57.3828,10.3349


,stationId,name,lat,lon
0,06141,Abed,54.8275,11.3292
1,06154,Brandelev,55.2075,11.8605
2,06136,Tystofte,55.2465,11.3285
3,06135,Flakkebjerg,55.3224,11.3879
4,06170,Roskilde Lufthavn,55.5867,12.1366
5,06180,Københavns Lufthavn,55.6140,12.6455
6,06186,Landbohøjskolen,55.6814,12.5403
7,06156,Holbæk,55.7154,11.7088
8,06181,Jægersborg,55.7664,12.5263
9,06169,Gniben,56.0067,11.2805


In [97]:
from pathlib import Path

out_dir = Path("raw/weather_station_selection")
out_dir.mkdir(parents=True, exist_ok=True)

dk1_selected.to_csv(out_dir / "stations_DK1.csv", index=False)
dk2_selected.to_csv(out_dir / "stations_DK2.csv", index=False)

print("Saved:", out_dir)

Saved: data/weather_station_selection


In [98]:
DK1_STATIONS = dk1_selected["stationId"].astype(str).tolist()
DK2_STATIONS = dk2_selected["stationId"].astype(str).tolist()

print("DK1 stationIds:", DK1_STATIONS)
print("DK2 stationIds:", DK2_STATIONS)

DK1 stationIds: ['06118', '06093', '06088', '06104', '06102', '06039', '06072', '06049', '06051', '06032']
DK2 stationIds: ['06141', '06154', '06136', '06135', '06170', '06180', '06186', '06156', '06181', '06169']


#### Extraxt data from stations

In [99]:
import requests, time
import pandas as pd

BASE_URL = "https://opendataapi.dmi.dk/v2/metObs/collections/observation/items"
PARAM_ID = "temp_dry"

DK1_STATIONS = dk1_selected["stationId"].astype(str).tolist()
DK2_STATIONS = dk2_selected["stationId"].astype(str).tolist()
ZONE_MAP = {sid: "DK1" for sid in DK1_STATIONS} | {sid: "DK2" for sid in DK2_STATIONS}
ALL_STATIONS = DK1_STATIONS + DK2_STATIONS

session = requests.Session()

def fetch_station_window(station_id, start_iso, end_iso, limit=10000):
    rows, offset = [], 0
    while True:
        params = {
            "stationId": station_id,
            "parameterId": PARAM_ID,
            "datetime": f"{start_iso}/{end_iso}",
            "limit": limit,
            "offset": offset
        }
        r = session.get(BASE_URL, params=params, timeout=60)
        r.raise_for_status()
        feats = r.json().get("features", [])
        if not feats:
            break

        for f in feats:
            p = f.get("properties", {})
            rows.append({
                "stationId": str(p.get("stationId")),
                "observed":  p.get("observed"),
                "TempC":     p.get("value"),
                "PriceArea": ZONE_MAP[station_id]
            })

        got = len(feats)
        offset += got
        if got < limit:
            break
        time.sleep(0.05)

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["observed"] = pd.to_datetime(df["observed"], utc=True, errors="coerce")
    df["TempC"] = pd.to_numeric(df["TempC"], errors="coerce")
    return df.dropna(subset=["observed","TempC"])

# one day, end-exclusive
start = pd.Timestamp("2021-01-15 00:00", tz="UTC")
end = start + pd.Timedelta(days=1) - pd.Timedelta(seconds=1)

start_iso = start.strftime("%Y-%m-%dT%H:%M:%SZ")
end_iso   = end.strftime("%Y-%m-%dT%H:%M:%SZ")

frames = []
for sid in ALL_STATIONS:
    df_s = fetch_station_window(sid, start_iso, end_iso)
    if not df_s.empty:
        frames.append(df_s)

df = pd.concat(frames, ignore_index=True)

# hourly aggregation
df["HourUTC"] = df["observed"].dt.floor("h")
df_station_hour = df.groupby(["HourUTC","PriceArea","stationId"], as_index=False)["TempC"].mean()
df_zone = df_station_hour.groupby(["HourUTC","PriceArea"], as_index=False)["TempC"].mean()

print("Stations with data:", df["stationId"].nunique(), "(expected near 20)")
print("Zone-hour rows:", len(df_zone), "(expected 48)")
print("Unique hours:", df_zone["HourUTC"].nunique(), "(expected 24)")
display(df_zone.head(10))

Stations with data: 19 (expected near 20)
Zone-hour rows: 48 (expected 48)
Unique hours: 24 (expected 24)


,HourUTC,PriceArea,TempC
0,2021-01-15 00:00:00+00:00,DK1,-1.838889
1,2021-01-15 00:00:00+00:00,DK2,-1.250000
2,2021-01-15 01:00:00+00:00,DK1,-2.012963
3,2021-01-15 01:00:00+00:00,DK2,-1.316667
4,2021-01-15 02:00:00+00:00,DK1,-2.224074
5,2021-01-15 02:00:00+00:00,DK2,-1.421667
6,2021-01-15 03:00:00+00:00,DK1,-2.524074
7,2021-01-15 03:00:00+00:00,DK2,-1.665000
8,2021-01-15 04:00:00+00:00,DK1,-2.622222
9,2021-01-15 04:00:00+00:00,DK2,-2.096667


In [100]:
# DATA FETCH — DMI temperature (temp_dry), monthly 2021–2025
# Saves:
# 1) Station-level: data/temp_station_hourly_monthly/temp_station_YYYY_MM.parquet
# 2) Zone-average:  data/temp_zone_hourly_monthly/temp_zone_YYYY_MM.parquet

import time
import requests
import pandas as pd
from pathlib import Path

# ---------- SETTINGS ----------
TEMP_PARAM = "temp_dry"
LON_SPLIT = 10.8
EXCLUDE_NAME_REGEX = "fyr|fyrskib|rev"   # avoid lighthouse/offshore-ish
RANDOM_STATE = 42

BASE_URL = "https://opendataapi.dmi.dk/v2/metObs/collections/observation/items"
session = requests.Session()

# ---------- AUTO-SELECT STATIONS THAT SUPPORT TEMP_PARAM ----------
# assumes you already have `stations` dataframe with columns:
# stationId, name, parameterId(list), lon, lat (from earlier step)

def select_spread_temp_stations(stations_df, zone, n=10):
    df = stations_df.copy()
    df["zone"] = df["lon"].apply(lambda x: "DK1" if x < LON_SPLIT else "DK2")

    df = df[df["parameterId"].apply(lambda xs: isinstance(xs, list) and (TEMP_PARAM in xs))].copy()
    df = df[df["zone"] == zone].copy()
    df = df.drop_duplicates(subset=["stationId"])

    df = df[~df["name"].str.lower().str.contains(EXCLUDE_NAME_REGEX, na=False)]
    bornholm = (df["lon"] > 14.0) & (df["lat"] > 54.9) & (df["lat"] < 55.4)
    df = df[~bornholm]

    df = df.sort_values("lat").reset_index(drop=True)
    if len(df) < n:
        raise ValueError(f"Not enough stations in {zone} supporting {TEMP_PARAM}: {len(df)} < {n}")

    df["lat_bin"] = pd.qcut(df["lat"], q=n, duplicates="drop")
    sel = (
        df.groupby("lat_bin", group_keys=False)
          .apply(lambda x: x.sample(1, random_state=RANDOM_STATE))
          .reset_index(drop=True)
          .sort_values("lat")
          .reset_index(drop=True)
    )
    return sel[["stationId","name","lat","lon"]]

dk1_selected = select_spread_temp_stations(stations, "DK1", n=10)
dk2_selected = select_spread_temp_stations(stations, "DK2", n=10)

DK1_STATIONS = dk1_selected["stationId"].astype(str).tolist()
DK2_STATIONS = dk2_selected["stationId"].astype(str).tolist()
ZONE_MAP = {sid: "DK1" for sid in DK1_STATIONS} | {sid: "DK2" for sid in DK2_STATIONS}
ALL_STATIONS = DK1_STATIONS + DK2_STATIONS

print("Selected stations:", len(ALL_STATIONS), "| DK1:", len(DK1_STATIONS), "| DK2:", len(DK2_STATIONS))

# ---------- OUTPUT DIRS ----------
raw_dir = Path("raw/temp_station_hourly_monthly")
zone_dir = Path("raw/temp_zone_hourly_monthly")
raw_dir.mkdir(parents=True, exist_ok=True)
zone_dir.mkdir(parents=True, exist_ok=True)

def fetch_station_month(station_id, start_iso, end_iso, limit=10000, sleep_s=0.05):
    rows, offset = [], 0
    while True:
        params = {
            "stationId": station_id,
            "parameterId": TEMP_PARAM,
            "datetime": f"{start_iso}/{end_iso}",
            "limit": limit,
            "offset": offset
        }
        r = session.get(BASE_URL, params=params, timeout=60)
        r.raise_for_status()
        feats = r.json().get("features", [])
        if not feats:
            break

        for f in feats:
            p = f.get("properties", {})
            rows.append({
                "stationId": str(p.get("stationId")),
                "observed":  p.get("observed"),
                "TempC":     p.get("value"),
                "PriceArea": ZONE_MAP[station_id]
            })

        got = len(feats)
        offset += got
        if got < limit:
            break

        time.sleep(sleep_s)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df["observed"] = pd.to_datetime(df["observed"], utc=True, errors="coerce")
    df["TempC"] = pd.to_numeric(df["TempC"], errors="coerce")
    df = df.dropna(subset=["observed","TempC"])
    return df

def fetch_month(year, month):
    start = pd.Timestamp(year=year, month=month, day=1, tz="UTC")
    end = start + pd.offsets.MonthBegin(1) - pd.Timedelta(seconds=1)  # end-exclusive

    start_iso = start.strftime("%Y-%m-%dT%H:%M:%SZ")
    end_iso   = end.strftime("%Y-%m-%dT%H:%M:%SZ")

    raw_path  = raw_dir / f"temp_station_{year}_{month:02d}.parquet"
    zone_path = zone_dir / f"temp_zone_{year}_{month:02d}.parquet"

    if raw_path.exists() and zone_path.exists():
        return

    frames = []
    for sid in ALL_STATIONS:
        df_s = fetch_station_month(sid, start_iso, end_iso)
        if not df_s.empty:
            frames.append(df_s)

    if not frames:
        print(f"Skipped (no data): {year}-{month:02d}")
        return

    df = pd.concat(frames, ignore_index=True)

    # hourly aggregation (critical)
    df["HourUTC"] = df["observed"].dt.floor("h")

    # average within station-hour first (handles multiple obs/hour)
    df_station_hour = (
        df.groupby(["HourUTC","PriceArea","stationId"], as_index=False)["TempC"].mean()
          .sort_values(["HourUTC","PriceArea","stationId"])
          .reset_index(drop=True)
    )
    df_station_hour.to_parquet(raw_path, index=False)

    # then average across stations to zone-hour
    df_zone = (
        df_station_hour.groupby(["HourUTC","PriceArea"], as_index=False)["TempC"].mean()
          .sort_values(["HourUTC","PriceArea"])
          .reset_index(drop=True)
    )
    df_zone.to_parquet(zone_path, index=False)

    print(f"Saved {year}-{month:02d} | station-hour rows: {len(df_station_hour)} | zone-hour rows: {len(df_zone)}")

# ---------- RUN ----------
for y in range(2021, 2026):
    for m in range(1, 13):
        fetch_month(y, m)

print("Done.")
print("Station-hour:", raw_dir)
print("Zone-hour:", zone_dir)

/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/1502318841.py:42: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("lat_bin", group_keys=False)
/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/1502318841.py:43: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(1, random_state=RANDOM_STATE))
/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/1502318841.py:42: FutureWarning: The default of observed=False is deprecated and will be changed t

Selected stations: 20 | DK1: 10 | DK2: 10
Saved 2021-01 | station-hour rows: 14057 | zone-hour rows: 1488
Saved 2021-02 | station-hour rows: 12305 | zone-hour rows: 1344
Saved 2021-03 | station-hour rows: 13763 | zone-hour rows: 1488
Saved 2021-04 | station-hour rows: 13680 | zone-hour rows: 1440
Saved 2021-05 | station-hour rows: 14064 | zone-hour rows: 1488
Saved 2021-06 | station-hour rows: 13428 | zone-hour rows: 1440
Saved 2021-07 | station-hour rows: 13962 | zone-hour rows: 1488
Saved 2021-08 | station-hour rows: 14056 | zone-hour rows: 1488
Saved 2021-09 | station-hour rows: 13435 | zone-hour rows: 1440
Saved 2021-10 | station-hour rows: 13876 | zone-hour rows: 1488
Saved 2021-11 | station-hour rows: 13607 | zone-hour rows: 1440
Saved 2021-12 | station-hour rows: 13977 | zone-hour rows: 1488
Saved 2022-01 | station-hour rows: 14044 | zone-hour rows: 1488
Saved 2022-02 | station-hour rows: 12768 | zone-hour rows: 1344
Saved 2022-03 | station-hour rows: 13690 | zone-hour rows: 148

##### Diagnostics

In [101]:
import pandas as pd
from pathlib import Path

files = sorted(Path("raw/temp_zone_hourly_monthly").glob("temp_zone_*.parquet"))
print("Files:", len(files))

total_rows = 0
tmin, tmax = None, None

for f in files:
    df = pd.read_parquet(f, columns=["HourUTC"])
    total_rows += len(df)
    mn = df["HourUTC"].min()
    mx = df["HourUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("Total rows:", total_rows)
print("Start HourUTC:", tmin)
print("End HourUTC:", tmax)

Files: 60
Total rows: 87648
Start HourUTC: 2021-01-01 00:00:00+00:00
End HourUTC: 2025-12-31 23:00:00+00:00


In [102]:
# DATA MANUAL — DMI Temperature (Zone-hour, DK1/DK2)
# Dataset: data/temp_zone_hourly_monthly/temp_zone_YYYY_MM.parquet
# Prints: coverage, schema, dtypes, missingness, uniqueness, sanity checks, snippet.

import pandas as pd
from pathlib import Path

DATASET_NAME = "Temperature (DMI metObs) — zone-hour average (DK1/DK2)"
DATA_DIR = Path("raw/temp_zone_hourly_monthly")
PATTERN = "temp_zone_*.parquet"

files = sorted(DATA_DIR.glob(PATTERN))
assert files, f"No files found in {DATA_DIR} matching {PATTERN}"

# --- Load one sample file for schema/dtypes ---
sample_file = files[0]
df_sample = pd.read_parquet(sample_file)

print("="*80)
print("DATASET:", DATASET_NAME)
print("DIRECTORY:", DATA_DIR)
print("FILES:", len(files))
print("SAMPLE FILE:", sample_file.name)
print("="*80)

# --- Schema ---
print("\nCOLUMNS:")
for c in df_sample.columns:
    print(" -", c)

print("\nDTYPES (sample file):")
print(df_sample.dtypes)

# --- Coverage across all files ---
total_rows = 0
tmin, tmax = None, None

for f in files:
    df_t = pd.read_parquet(f, columns=["HourUTC"])
    total_rows += len(df_t)
    mn = df_t["HourUTC"].min()
    mx = df_t["HourUTC"].max()
    tmin = mn if tmin is None else min(tmin, mn)
    tmax = mx if tmax is None else max(tmax, mx)

print("\nCOVERAGE:")
print(" - Total rows (all files):", total_rows)
print(" - Start HourUTC:", tmin)
print(" - End HourUTC:", tmax)

# --- Key integrity checks (sample file) ---
print("\nINTEGRITY CHECKS (sample file):")

key_cols = ["HourUTC", "PriceArea"]
dup_keys = df_sample.duplicated(key_cols).sum()
print(" - Duplicate (HourUTC, PriceArea) rows:", int(dup_keys))

if "PriceArea" in df_sample.columns:
    print(" - PriceArea unique values:", sorted(df_sample["PriceArea"].dropna().unique()))

# --- Missingness (sample file) ---
print("\nMISSINGNESS (sample file):")
na_counts = df_sample.isna().sum()
for c in df_sample.columns:
    print(f" - {c}: {int(na_counts[c])} missing")

# --- Temperature sanity checks (sample file) ---
temp_col = "TempC"
if temp_col in df_sample.columns:
    s = df_sample[temp_col]
    print(f"\nBASIC STATS ({temp_col}) (sample file):")
    print(" - min:", float(s.min()))
    print(" - max:", float(s.max()))
    print(" - mean:", float(s.mean()))
    print(" - zeros:", int((s == 0).sum()))
    print(" - negatives:", int((s < 0).sum()))
    print(" - missing:", int(s.isna().sum()))

# --- Rows per hour sanity check (sample file) -> should be 2 (DK1, DK2)
rows_per_hour = df_sample.groupby("HourUTC").size()
print("\nROWS PER HOUR (sample file):")
print(" - min:", int(rows_per_hour.min()))
print(" - max:", int(rows_per_hour.max()))
print(" - value counts:")
display(rows_per_hour.value_counts().head(10))

# --- Snippet ---
print("\nSNIPPET (first 10 rows of sample file):")
display(df_sample.head(10))

DATASET: Temperature (DMI metObs) — zone-hour average (DK1/DK2)
DIRECTORY: data/temp_zone_hourly_monthly
FILES: 60
SAMPLE FILE: temp_zone_2021_01.parquet

COLUMNS:
 - HourUTC
 - PriceArea
 - TempC

DTYPES (sample file):
HourUTC      datetime64[ns, UTC]
PriceArea                 object
TempC                    float64
dtype: object

COVERAGE:
 - Total rows (all files): 87648
 - Start HourUTC: 2021-01-01 00:00:00+00:00
 - End HourUTC: 2025-12-31 23:00:00+00:00

INTEGRITY CHECKS (sample file):
 - Duplicate (HourUTC, PriceArea) rows: 0
 - PriceArea unique values: ['DK1', 'DK2']

MISSINGNESS (sample file):
 - HourUTC: 0 missing
 - PriceArea: 0 missing
 - TempC: 0 missing

BASIC STATS (TempC) (sample file):
 - min: -5.885185185185185
 - max: 6.9799999999999995
 - mean: 0.8946276086113955
 - zeros: 0
 - negatives: 432
 - missing: 0

ROWS PER HOUR (sample file):
 - min: 2
 - max: 2
 - value counts:


2    744
Name: count, dtype: int64


SNIPPET (first 10 rows of sample file):


,HourUTC,PriceArea,TempC
0,2021-01-01 00:00:00+00:00,DK1,1.022222
1,2021-01-01 00:00:00+00:00,DK2,2.358333
2,2021-01-01 01:00:00+00:00,DK1,0.950000
3,2021-01-01 01:00:00+00:00,DK2,2.371667
4,2021-01-01 02:00:00+00:00,DK1,0.961111
5,2021-01-01 02:00:00+00:00,DK2,2.485000
6,2021-01-01 03:00:00+00:00,DK1,0.931481
7,2021-01-01 03:00:00+00:00,DK2,2.526667
8,2021-01-01 04:00:00+00:00,DK1,1.016667
9,2021-01-01 04:00:00+00:00,DK2,2.306667


### API fetch: Price GAS

In [121]:
import pandas as pd

gas = pd.read_excel("raw/GasPricesDatastream.xlsx")
coal = pd.read_excel("raw/CoalPricesDatastream.xlsx")
carbon = pd.read_excel("raw/CarbonPricesDatastream.xlsx")

print(gas.head())
print(coal.head())
print(carbon.head())

        Date EEX-TTF-D (TRDPRC_1)
0        NaT                Close
1 2026-02-20                33.43
2 2026-02-18               30.334
3 2026-02-17               30.161
4 2026-02-16               30.986
        Date TRAPI2Mc1 (TRDPRC_1)
0        NaT                Close
1 2026-02-19               112.75
2 2026-02-18               108.15
3 2026-02-12               104.13
4 2026-02-05                 99.9
        Date CFI2Zc1 (TRDPRC_1)
0        NaT              Close
1 2026-02-19              71.34
2 2026-02-18              71.57
3 2026-02-17              70.08
4 2026-02-16              69.18


In [122]:
for name, df in {
    "Gas": gas,
    "Coal": coal,
    "Carbon": carbon
}.items():
    
    # Assume first column is date
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    
    print(f"\n{name}")
    print("Start:", df[date_col].min())
    print("End:  ", df[date_col].max())
    print("Rows: ", len(df))


Gas
Start: 2020-01-01 00:00:00
End:   2026-02-20 00:00:00
Rows:  1571

Coal
Start: 2016-02-19 00:00:00
End:   2026-02-19 00:00:00
Rows:  1724

Carbon
Start: 2016-02-19 00:00:00
End:   2026-02-19 00:00:00
Rows:  2574


In [123]:
import pandas as pd
from pathlib import Path

START = "2020-01-01"
END   = "2025-12-31"

def clean_daily_control(input_path, value_name, output_name):

    df = pd.read_excel(input_path)

    # Rename first two columns
    df = df.rename(columns={
        df.columns[0]: "Date",
        df.columns[1]: value_name
    })

    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")

    # Restrict period
    df = df[(df["Date"] >= START) & (df["Date"] <= END)]

    # Expand to full daily calendar
    df = df.set_index("Date").asfreq("D")
    df[value_name] = df[value_name].ffill()

    df = df.reset_index()

    out_dir = Path("raw/controls")
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / output_name
    df.to_parquet(out_path, index=False)

    print("Saved:", out_path)
    print("Rows:", len(df))
    print("Start:", df["Date"].min(), "| End:", df["Date"].max())

    return df

In [124]:
gas = clean_daily_control(
    "raw/GasPricesDatastream.xlsx",
    "Gas_EUR_MWh",
    "gas_daily_2020_2025.parquet"
)

coal = clean_daily_control(
    "raw/CoalPricesDatastream.xlsx",
    "Coal_USD_ton",
    "coal_daily_2020_2025.parquet"
)

carbon = clean_daily_control(
    "raw/CarbonPricesDatastream.xlsx",
    "EUA_EUR_ton",
    "carbon_daily_2020_2025.parquet"
)

Saved: data/controls/gas_daily_2020_2025.parquet
Rows: 2192
Start: 2020-01-01 00:00:00 | End: 2025-12-31 00:00:00
Saved: data/controls/coal_daily_2020_2025.parquet
Rows: 2174
Start: 2020-01-06 00:00:00 | End: 2025-12-18 00:00:00
Saved: data/controls/carbon_daily_2020_2025.parquet
Rows: 2191
Start: 2020-01-02 00:00:00 | End: 2025-12-31 00:00:00


/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/2364318159.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[value_name] = df[value_name].ffill()
/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/2364318159.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[value_name] = df[value_name].ffill()
/var/folders/2t/ffyv8q1x20j81h8dg304mct40000gn/T/ipykernel_68631/2364318159.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.in

##### Diagnostics

In [125]:
import pandas as pd
from pathlib import Path

def data_manual_daily(path, value_col):
    df = pd.read_parquet(path)

    print("="*70)
    print(f"DATASET: {path.name}")
    print("="*70)

    print("\nSTRUCTURE")
    print("Rows:", len(df))
    print("Columns:", list(df.columns))

    print("\nTIME COVERAGE")
    print("Start:", df["Date"].min())
    print("End:  ", df["Date"].max())
    print("Unique days:", df["Date"].nunique())

    print("\nBASIC STATS")
    print("Min:     ", df[value_col].min())
    print("Max:     ", df[value_col].max())
    print("Mean:    ", df[value_col].mean())
    print("Zeros:   ", (df[value_col] == 0).sum())
    print("Negatives:", (df[value_col] < 0).sum())
    print("Missing: ", df[value_col].isna().sum())

    print("\nWEEKEND CHECK")
    weekend_rows = df[df["Date"].dt.weekday >= 5]
    print("Weekend observations:", len(weekend_rows))

    print("\nHEAD")
    print(df.head())

    print("\nTAIL")
    print(df.tail())
    print("\n")


# ------------------------------------------------------------------
# Run for all three
# ------------------------------------------------------------------

base = Path("raw/controls")

data_manual_daily(base / "gas_daily_2020_2025.parquet", "Gas_EUR_MWh")
data_manual_daily(base / "coal_daily_2020_2025.parquet", "Coal_USD_ton")
data_manual_daily(base / "carbon_daily_2020_2025.parquet", "EUA_EUR_ton")

DATASET: gas_daily_2020_2025.parquet

STRUCTURE
Rows: 2192
Columns: ['Date', 'Gas_EUR_MWh']

TIME COVERAGE
Start: 2020-01-01 00:00:00
End:   2025-12-31 00:00:00
Unique days: 2192

BASIC STATS
Min:      3.609
Max:      317.653
Mean:     50.25049863138686
Zeros:    0
Negatives: 0
Missing:  0

WEEKEND CHECK
Weekend observations: 626

HEAD
        Date  Gas_EUR_MWh
0 2020-01-01       12.216
1 2020-01-02       12.216
2 2020-01-03       12.539
3 2020-01-04       12.539
4 2020-01-05       12.539

TAIL
           Date  Gas_EUR_MWh
2187 2025-12-27       28.165
2188 2025-12-28       28.165
2189 2025-12-29       28.421
2190 2025-12-30       27.814
2191 2025-12-31       28.049


DATASET: coal_daily_2020_2025.parquet

STRUCTURE
Rows: 2174
Columns: ['Date', 'Coal_USD_ton']

TIME COVERAGE
Start: 2020-01-06 00:00:00
End:   2025-12-18 00:00:00
Unique days: 2174

BASIC STATS
Min:      38.68
Max:      462.0
Mean:     130.5880496780129
Zeros:    0
Negatives: 0
Missing:  0

WEEKEND CHECK
Weekend observatio

### API FETCH: Generation by Production Type and Exchange (GenerationProdTypeExchange)

In [1]:
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.energidataservice.dk/dataset/GenerationProdTypeExchange"
out_dir = Path("raw/generation_prod_type_exchange")
out_dir.mkdir(parents=True, exist_ok=True)

AREAS = ["DK1", "DK2"]
LIMIT = 100_000

def fetch_generation_year(year: int, areas=("DK1", "DK2")):
    saved, skipped = 0, 0
    for area in areas:
        out_file = out_dir / f"generation_{area}_{year}.parquet"
        if out_file.exists():
            skipped += 1
            continue

        start = f"{year}-01-01T00:00"
        end   = f"{year + 1}-01-01T00:00"
        records = []
        offset = 0

        while True:
            params = {
                "start":  start,
                "end":    end,
                "filter": f'{{"PriceArea":"{area}"}}',
                "limit":  LIMIT,
                "offset": offset,
                "sort":   "TimeUTC asc",
            }
            r = requests.get(BASE_URL, params=params, timeout=60)
            r.raise_for_status()
            batch = r.json().get("records", [])
            if not batch:
                break
            records.extend(batch)
            if len(batch) < LIMIT:
                break
            offset += LIMIT

        if records:
            df = pd.DataFrame(records)
            for col in ["TimeUTC", "TimeDK"]:
                if col in df.columns:
                    df[col] = pd.to_datetime(df[col])
            df.to_parquet(out_file, index=False)
            print(f"  Saved {len(df):,} rows → {out_file.name}")
            saved += 1
        else:
            print(f"  No data for {area} {year}")
    return saved, skipped


saved_total, skipped_total = 0, 0
for y in range(2021, 2026):
    s, k = fetch_generation_year(y)
    saved_total += s
    skipped_total += k

print(f"\nDone. Saved {saved_total} files, skipped {skipped_total} existing.")

  Saved 8,760 rows → generation_DK1_2021.parquet
  Saved 8,760 rows → generation_DK2_2021.parquet
  Saved 8,760 rows → generation_DK1_2022.parquet
  Saved 8,760 rows → generation_DK2_2022.parquet
  Saved 8,760 rows → generation_DK1_2023.parquet
  Saved 8,760 rows → generation_DK2_2023.parquet
  Saved 8,784 rows → generation_DK1_2024.parquet
  Saved 8,784 rows → generation_DK2_2024.parquet
  Saved 8,760 rows → generation_DK1_2025.parquet
  Saved 8,760 rows → generation_DK2_2025.parquet

Done. Saved 10 files, skipped 0 existing.


##### Diagnostics

In [2]:
# DATA MANUAL — GenerationProdTypeExchange (Hourly generation by production type + exchange, DK1/DK2)
# Prints: file coverage, columns, row counts, and a sample row

import pandas as pd
from pathlib import Path

files = sorted(Path("raw/generation_prod_type_exchange").glob("*.parquet"))
print(f"Files found: {len(files)}\n")

for f in files:
    df = pd.read_parquet(f)
    t_min = df["TimeUTC"].min() if "TimeUTC" in df.columns else "?"
    t_max = df["TimeUTC"].max() if "TimeUTC" in df.columns else "?"
    print(f"{f.name}: {len(df):,} rows | {t_min} → {t_max}")

if files:
    df_sample = pd.read_parquet(files[0])
    print("\nColumns:", df_sample.columns.tolist())
    print("\nSample row:")
    print(df_sample.iloc[0].to_string())

Files found: 10

generation_DK1_2021.parquet: 8,760 rows | 2020-12-31 23:00:00 → 2021-12-31 22:00:00
generation_DK1_2022.parquet: 8,760 rows | 2021-12-31 23:00:00 → 2022-12-31 22:00:00
generation_DK1_2023.parquet: 8,760 rows | 2022-12-31 23:00:00 → 2023-12-31 22:00:00
generation_DK1_2024.parquet: 8,784 rows | 2023-12-31 23:00:00 → 2024-12-31 22:00:00
generation_DK1_2025.parquet: 8,760 rows | 2024-12-31 23:00:00 → 2025-12-31 22:00:00
generation_DK2_2021.parquet: 8,760 rows | 2020-12-31 23:00:00 → 2021-12-31 22:00:00
generation_DK2_2022.parquet: 8,760 rows | 2021-12-31 23:00:00 → 2022-12-31 22:00:00
generation_DK2_2023.parquet: 8,760 rows | 2022-12-31 23:00:00 → 2023-12-31 22:00:00
generation_DK2_2024.parquet: 8,784 rows | 2023-12-31 23:00:00 → 2024-12-31 22:00:00
generation_DK2_2025.parquet: 8,760 rows | 2024-12-31 23:00:00 → 2025-12-31 22:00:00

Columns: ['TimeUTC', 'TimeDK', 'PriceArea', 'Version', 'GrossCon', 'OffshoreWindPower', 'OnshoreWindPower', 'HydroPower', 'SolarPower', 'Solar